In [1]:
# =====================================================================================
# Project: PTB-XL Metadata Audit  -  v0.7 EXPLORATORY development matrix on fold 9
# Code version: v0.7
# Date: 2026-10-02
# Locked by: final_run_protocol.json -> stage_plan["v0.7"] (this file has priority over any prompt)
#
# Purpose:
#   STEP 0  snapshot PREREG_DIR; verify the five locked documents (self_sha256 + file sha256) and
#           their parent chain; check the five OpenTimestamps .ots files exist (hash + size only);
#           verify the v0.4 / v0.5 / v0.6 manifests, the input CSVs, the v0.4 signal cache, the 9
#           reused v0.5 run files; load the recipe FROM dev_protocol.json; rebuild TRAINING_DETAILS
#           verbatim from v0.5 and assert it equals v0.5_training_details.json["details"]; exec the
#           reference implementation read FROM final_run_protocol.json (sha256-asserted)
#   A  environment regression check (v0.5 Part A verbatim)
#   B  the 3 reused v0.5 cells (no retraining): AUROC recomputed from the saved fold-9
#      probabilities and asserted equal to lr_selection.json / v0.5_manifest.json
#   C  OUT_DIR\v0.7_exploratory_analysis_plan.json (write-once) BEFORE the first training run
#   D  15 new training runs (2 archs x 3 arms x 3 seeds minus the 3 reused), lr 0.01, folds 1-8,
#      ONE fold-9 evaluation at the end of epoch 50; append-only attempt log; crash/re-run rule
#   E  exploratory analysis E1-E7 (fold 9 only; numbers and intervals only; no decision)
#
# STATUS: EXPLORATORY. No setting may be chosen, changed or re-run based on any v0.7 number.
# Nothing in PREREG_DIR is written or modified (snapshot asserted identical at the end).
# FOLD 10: never opened. Working set = strat_fold <= 9 only (asserted); signals only from the
# v0.4 cache of folds 1-9 (sha256-verified); wfdb is NOT imported.
#
# REUSE (verbatim from v0.5): helpers, verify_self_sha256, canonical hashing, write_once_json,
# recipe parsing, label derivation, metadata preprocessing, folds 1-8 encoder, META matrices,
# Part A, GPU setup, lead normalization (folds 1-8), MODELS block, METADATA_FUSION_DESCRIPTION,
# RandomCropDataset, TRAINING_DETAILS, run naming, predict_fold9, load_saved_probs,
# macro_auroc_from_probs. Only CODE_VERSION / OUT_DIR differ.
#
# DOCUMENTED CHANGES vs v0.5 (all deliberate):
#   (1) the FORBIDDEN_ARM guard of train_run() is REMOVED (final_run_protocol stage_plan v0.7
#       trains signal_plus_all);
#   (2) lr = SELECTED_LR[arch] (0.01), no lr loop;
#   (3) end of training: predict_fold9 -> save state_dict, train-loss CSV, then the fold-9
#       probability CSV (written to a .partial file and renamed, so "probability CSV exists"
#       <=> "fold-9 evaluation was computed and all three files are complete") -> log
#       "completed" with the hashes -> only then compute AUROC;
#   (4) assert model.hidden[0].in_features == 256 / 259 / 338 and OneCycle total_steps == 6850;
#   (5) after counting fold-10 rows in the source CSV, df_all is restricted to folds 1-9 (no
#       fold-10 value stays in memory; no number changes).
#
# RE-RUN BEHAVIOUR (crash_and_rerun_rules["v0.7"]; log = OUT_DIR\v0.7_attempt_log.jsonl):
#   - probability CSV exists -> COMPLETE, never re-run; hashes verified against the log
#     ("completed_recovered" + WARN if the log has no completed entry);
#   - no probability CSV -> partial files deleted (logged) and the run is re-trained from scratch
#     with the same seed as attempt n+1; at most 2 attempts, then STOP (no setting changed).
#   - all analysis outputs (tables, bootstrap, manifest) are regenerated from saved files.
#
# Outputs: OUT_DIR = <PROJECT_ROOT>\outputs\v0.7
#   v0.7_env_regression.csv, v0.7_exploratory_analysis_plan.json (write-once), v0.7_attempt_log.jsonl,
#   per new run: v0.7_{arch}_{arm}_seed{seed}_lr0.01_fold9_probs.csv / _train_loss.csv / _state_dict.pt,
#   v0.7_runs_table.csv, v0.7_seed_summary.csv, v0.7_ensemble_table.csv,
#   v0.7_ensemble_{arch}_{arm}_fold9_probs.csv, v0.7_bootstrap_summary.csv,
#   v0.7_bootstrap_deltas_ensemble.csv, v0.7_manifest.json
# =====================================================================================

import os, sys, json, ast, hashlib, platform, time, random, math, re, gc, ntpath
from datetime import datetime
import numpy as np
import pandas as pd
import scipy
from scipy.stats import rankdata
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

_WFDB_AT_START = "wfdb" in sys.modules

pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

# ------------------------------- configuration ---------------------------------------
PROJECT_ROOT = r"D:\ptbxl-metadata-audit"
DATA_ROOT  = os.path.join(PROJECT_ROOT, "data", "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
OUT_DIR    = os.path.join(PROJECT_ROOT, "outputs", "v0.7")
V04_DIR    = os.path.join(PROJECT_ROOT, "outputs", "v0.4")
V05_DIR    = os.path.join(PROJECT_ROOT, "outputs", "v0.5")
V06_DIR    = os.path.join(PROJECT_ROOT, "outputs", "v0.6")
PREREG_DIR = os.path.join(PROJECT_ROOT, "preregistration")

SEAL_PATH             = os.path.join(PREREG_DIR, "pre_registration.json")
DEV_PROTOCOL_PATH     = os.path.join(PREREG_DIR, "dev_protocol.json")
TRAINING_DETAILS_PATH = os.path.join(PREREG_DIR, "v0.5_training_details.json")
LR_SELECTION_PATH     = os.path.join(PREREG_DIR, "lr_selection.json")
FINAL_PROTOCOL_PATH   = os.path.join(PREREG_DIR, "final_run_protocol.json")
DOC_PATHS = {"pre_registration": SEAL_PATH, "dev_protocol": DEV_PROTOCOL_PATH,
             "training_details": TRAINING_DETAILS_PATH, "lr_selection": LR_SELECTION_PATH,
             "final_run_protocol": FINAL_PROTOCOL_PATH}

V04_MANIFEST = os.path.join(V04_DIR, "v0.4_manifest.json")
V05_MANIFEST = os.path.join(V05_DIR, "v0.5_manifest.json")
V06_MANIFEST = os.path.join(V06_DIR, "v0.6_manifest.json")
SIGNALS_NPY = os.path.join(V04_DIR, "v0.4_signals_folds1to9_100hz.npy")
SIGNALS_INDEX = os.path.join(V04_DIR, "v0.4_signals_index.csv")
V04_LEAD_NORM_CSV = os.path.join(V04_DIR, "v0.4_lead_norm_params.csv")

CODE_VERSION = "v0.7"
PREFIX = CODE_VERSION + "_"
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

# --- the five locked documents: self_sha256 (canonical JSON) and file sha256 (bytes on disk) ---
EXPECTED_SELF_SHA256 = {
    "pre_registration":   "5183f54569f3358a5339b16a2c3c3be49f5c7798eabe6b0bb25946119c2ba469",
    "dev_protocol":       "c967ac38c8cc9382bf41fdd3785c8bbd730db196e349846b8052d856bbac5fc6",
    "training_details":   "c12c33a2f649984fb1b70a46445823c46b57d1b024d87655e16661789f04b46f",
    "lr_selection":       "c70cbfa65e0ab75321dfef9c64714a64705c60a967685d79a878a1d40450749d",
    "final_run_protocol": "834acddec288862d0418e9ee70204898c9d371788eed7eb362f4c0d63d305c6b",
}
EXPECTED_FILE_SHA256 = {
    "pre_registration":   "11dd446272654fc80d65b2fdd40259fa674bbb419a44fde84e589b6e297cde32",
    "dev_protocol":       "babf101c1d69b322daef6907175fdc4c17b5a53755a68f13a998eefa0148dc2d",
    "training_details":   "6480c191aeaed9ff1e6b1da5caa0c1e46908d453d30106a32beb7cf83678be11",
    "lr_selection":       "358d783b37bba6fcf61237a43817c1fd8594fe192f8ecdfca2c25b871a21415b",
    "final_run_protocol": "e1a5260e72d2e8b0d27dc709639468b7f9a48a7c8e2f8af2d663e105dbd329d4",
}
OTS_NOTE = "Bitcoin attestation of all five .ots files confirmed by the user on 2026-10-02 (not re-verified from Python)"

EXPECTED_V04_MANIFEST_SHA256 = "3781f90c4cf5a85ea612efabb1fd81cc61b77cbeb38a54cc644c10c66afe1d9b"
EXPECTED_V05_MANIFEST_SHA256 = "59968bfb50d8c919de0e8004ef878a44a115560de438b7e74582877495e95f87"
EXPECTED_V06_MANIFEST_SHA256 = "2cb7a1b023e03aad85bb11150ad38e203624300170bc0fa197014c727758c61a"
EXPECTED_REFERENCE_CODE_SHA256 = "48ac837c07880f0bd88a7bd01475e7da3edbeab46019f8288976884ad7662856"

EXPECTED_INPUT_HASHES = {
    "ptbxl_database.csv": "7600de9c1b27d181d850b3c6038a35d7c3ddb6bb33b702e3a20252a6859d216b",
    "scp_statements.csv": "ad05b0b1fcae83bb1230755ad9cfc7c96f303feddc08a4a9ad5bdc9ca63bac8f",
}
EXPECTED_SIGNALS_NPY_SHA256 = "17558f9d09b39b990d7f3a5ca2ea10e1527980288a49a6828c89b7c3df61745a"
EXPECTED_SIGNALS_INDEX_SHA256 = "06b49ecdafc6a74c9bdbcfa0177bb357eb845ece187b40e3a777496a8cf72203"
EXPECTED_LEAD_NORM_CSV_SHA256 = "07fb6d6dc8f0f1e2379fdd45b6462f1102518621eb242ba585fb8600acff35aa"

EXPECTED_FOLD9_POSITIVES = {"NORM": 955, "MI": 540, "STTC": 528, "CD": 495, "HYP": 268}
EXPECTED_N_RECORDS_FOLDS1TO9 = 19601
EXPECTED_ONEHOT_COUNTS = {"device": 13, "site": 52, "nurse": 14}
V05_AGE_SCALER_FOLDS1TO8 = {"mean": 59.7333, "scale": 16.8750}      # printed with 4 decimals in v0.5
AGE_SCALER_TOL = 5e-5

# Part A: v0.3 LogisticRegression seed-42 fold-9 macro-AUROC (6 decimals)
V03_LR_MACRO_AUROC = {"B1_physio": 0.688969, "B3_physio_plus_acquisition": 0.697165}
ENV_REGRESSION_TOL = 1e-5

# constants that dev_protocol.json defines (asserted equal to the file; the FILE values are used)
EXPECTED_EPOCHS = 50
EXPECTED_LR_GRID = [1e-3, 3e-3, 1e-2]
EXPECTED_CROP = 250
EXPECTED_EVAL_STRIDE = 125
EXPECTED_WINDOW_STARTS = [0, 125, 250, 375, 500, 625, 750]
EXPECTED_WEIGHT_DECAY = 1e-2
EXPECTED_BATCH_SIZE = {"resnet1d_wang": 128, "inception1d": 128}
EXPECTED_SELECTED_LR = {"resnet1d_wang": 0.01, "inception1d": 0.01}

SEED = 42
FS = 100
N_SAMPLES = 1000
N_LEADS = 12
NUM_WORKERS = 0                   # Windows + Jupyter
EVAL_BATCH_SIZE = 512             # inference only; does not affect results (model.eval())
LR_TIE_TOL = 1e-4                 # appears in TRAINING_DETAILS (verbatim v0.5); not used for selection here
SKLEARN_CHECK_TOL = 1e-9

ARCHS = ["resnet1d_wang", "inception1d"]
ARMS = ["signal_only", "signal_plus_physio", "signal_plus_all"]
SEEDS = [42, 123, 2024]
EXPECTED_META_DIM = {"signal_only": 0, "signal_plus_physio": 3, "signal_plus_all": 82}
EXPECTED_HEAD_IN = {"signal_only": 256, "signal_plus_physio": 259, "signal_plus_all": 338}
EXPECTED_TOTAL_STEPS_DEV = 6850                  # 50 epochs x ceil(17418 / 128) = 50 x 137
REUSED_CELLS = [("resnet1d_wang", "signal_plus_physio", 42), ("inception1d", "signal_plus_physio", 42),
                ("resnet1d_wang", "signal_only", 42)]
EXPECTED_NEW_RUNS = {"resnet1d_wang": 7, "inception1d": 8}
MAX_ATTEMPTS = 2
REUSE_TOL = 1e-12

# Part E
N_BOOT = 2000
BOOT_SEED = 7
EXPECTED_FOLD9_PATIENTS = 1942
NAN_DRAW_FLAG_FRACTION = 0.01
COMPARISONS = {"C1": ("signal_plus_physio", "signal_plus_all"),      # delta = B - A
               "C2": ("signal_only", "signal_plus_physio")}
PLAUSIBILITY_MIN_AUROC = 0.90
LOSS_OUTLIER_FACTOR = 2.0
BIAS_NOTE = ("lr was selected on fold 9 with signal_plus_physio (seed 42): on fold 9, C2 (physio - signal_only) "
             "is biased UPWARD and C1 (all - physio) is biased DOWNWARD. Fold 10 (v0.9) is not affected. EXPLORATORY.")

_root_abs = os.path.abspath(DATA_ROOT)
for _p in (OUT_DIR, PREREG_DIR):
    assert not os.path.abspath(_p).lower().startswith(_root_abs.lower()), \
        "output path is inside the dataset folder - aborting: " + _p

RUN_STARTED = datetime.now().astimezone().isoformat()
OUTPUT_FILES = []        # paths of produced files (for the manifest)
WARNINGS = []


def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def save_csv(obj, name, index=True):
    p = os.path.join(OUT_DIR, name)
    obj.to_csv(p, index=index, encoding="utf-8-sig")
    OUTPUT_FILES.append(p)
    print("[saved] " + p)
    return p


def hdr(t):
    print("\n" + "=" * 100)
    print(t)
    print("=" * 100)


def warn(msg):
    WARNINGS.append(msg)
    print("[WARN] " + msg)


def note(msg):
    print("[NOTE] " + msg)


def _json_default(o):
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.floating):
        return float(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


def to_level_string(series, missing="__MISSING__"):
    """Human-readable, zero-padded categorical level for numeric columns (e.g.
    '007' for site) so sorting and printed feature names stay legible."""
    if pd.api.types.is_numeric_dtype(series):
        nn = series.dropna()
        integral = bool(len(nn) == 0 or np.all(np.isclose(nn.values, np.round(nn.values))))
        if integral:
            width = max(3, len(str(int(nn.max()))) if len(nn) else 3)
            return series.apply(lambda v: missing if pd.isna(v) else str(int(v)).zfill(width))
    return series.apply(lambda v: missing if pd.isna(v) else str(v).strip())


def encode_categorical(level_series, train_mask, col_name):
    """One-hot encode a pre-stringified categorical column. The category list is
    fixed from the training rows only, always includes an explicit '__MISSING__'
    level, and always includes a catch-all '__UNKNOWN__' level for anything seen
    only outside training (e.g. only in fold 9). Both splits therefore produce the
    exact same set of columns."""
    train_vals = level_series[train_mask]
    train_categories = sorted(set(train_vals.unique().tolist()) | {"__MISSING__"})
    known = set(train_categories)
    remapped = level_series.where(level_series.isin(known), other="__UNKNOWN__")
    categories = train_categories + ["__UNKNOWN__"]
    cat = pd.Categorical(remapped, categories=categories)
    dummies = pd.get_dummies(cat)
    dummies.columns = ["{}={}".format(col_name, c) for c in dummies.columns]
    dummies.index = level_series.index
    return dummies, categories


def canonical_json_bytes(d):
    """Canonical serialization used for self_sha256 (see SELF_SHA256_PROCEDURE)."""
    return json.dumps(d, sort_keys=True, ensure_ascii=False, separators=(",", ":"),
                      default=_json_default).encode("utf-8")


SELF_SHA256_PROCEDURE = (
    "Load the JSON file; remove the key 'self_sha256' (the key must be absent, not empty); "
    "serialize the remaining dict with json.dumps(d, sort_keys=True, ensure_ascii=False, "
    "separators=(',', ':')); compute SHA-256 of the UTF-8 bytes; the hex digest must equal "
    "the stored 'self_sha256'. (At write time numpy scalars were converted to plain JSON "
    "types first, so the loaded file hashes identically.)"
)


def verify_self_sha256(path):
    """Return (stored, recomputed, loaded_dict) for a self-hashed JSON document."""
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    stored = d.pop("self_sha256", None)
    recomputed = hashlib.sha256(canonical_json_bytes(d)).hexdigest()
    d["self_sha256"] = stored
    return stored, recomputed, d


def fast_auc(y, s):
    """Binary AUROC via the Mann-Whitney U statistic (average-rank tie handling).
    Used instead of sklearn.metrics.roc_auc_score for speed inside the bootstrap
    loop; both compute the identical quantity."""
    y = np.asarray(y)
    n1 = int(y.sum())
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return float("nan")
    ranks = rankdata(np.asarray(s))
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))


def write_once_json(path, doc):
    """Normalize to plain JSON types, add self_sha256, exclusive-create, verify from disk."""
    doc = json.loads(json.dumps(doc, ensure_ascii=False, default=_json_default))
    assert "self_sha256" not in doc
    sha = hashlib.sha256(canonical_json_bytes(doc)).hexdigest()
    doc["self_sha256"] = sha
    with open(path, "x", encoding="utf-8") as f:          # refuses to overwrite
        json.dump(doc, f, ensure_ascii=False, indent=2)
    s_, r_, _ = verify_self_sha256(path)
    assert s_ == r_ == sha, "self_sha256 verification FAILED for " + path
    return sha


def plain(o):
    return json.loads(json.dumps(o, ensure_ascii=False, default=_json_default))


def manifest_entry(manifest, basename):
    """Return the unique 'outputs' entry of a project manifest whose file basename matches
    (works for full Windows paths and bare names)."""
    hits = [o for o in manifest["outputs"] if ntpath.basename(o["file"]) == basename]
    assert len(hits) == 1, "expected exactly one manifest entry for {}, found {}".format(basename, len(hits))
    return hits[0]


def load_verified_manifest(path, expected_sha, label):
    assert os.path.isfile(path), label + " not found: " + path + " - aborting (fix the path constant)."
    got = sha256_of(path)
    assert got == expected_sha, "{} file sha256 {} != expected {} - aborting.".format(label, got, expected_sha)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f), got


def prereg_snapshot():
    """Sorted (name, sha256) of every file directly in PREREG_DIR (incl. the .ots files)."""
    out = []
    for name in sorted(os.listdir(PREREG_DIR)):
        p = os.path.join(PREREG_DIR, name)
        if os.path.isfile(p):
            out.append((name, sha256_of(p)))
    return out


# =====================================================================================
# STEP 0 - INTEGRITY (nothing is written before it passes)
# =====================================================================================
hdr("STEP 0 - INTEGRITY")

# --- 1. PREREG_DIR snapshot (asserted identical at the end of the cell) ---
assert os.path.isdir(PREREG_DIR), "PREREG_DIR not found: " + PREREG_DIR
PREREG_SNAPSHOT_START = prereg_snapshot()
print("PREREG_DIR snapshot: {} files".format(len(PREREG_SNAPSHOT_START)))
for name, h in PREREG_SNAPSHOT_START:
    print("  {}  {}".format(h, name))

# --- 2. the five locked documents ---
DOCS, DOC_SELF, DOC_FILE = {}, {}, {}
for key, path in DOC_PATHS.items():
    assert os.path.isfile(path), key + " not found: " + path + " - aborting."
    _s, _r, _d = verify_self_sha256(path)
    assert _s == _r == EXPECTED_SELF_SHA256[key], \
        "{}: self_sha256 stored {} / recomputed {} / expected {} - aborting.".format(key, _s, _r, EXPECTED_SELF_SHA256[key])
    _f = sha256_of(path)
    assert _f == EXPECTED_FILE_SHA256[key], "{}: file sha256 {} != {} - aborting.".format(key, _f, EXPECTED_FILE_SHA256[key])
    DOCS[key], DOC_SELF[key], DOC_FILE[key] = _d, _s, _f
    print("{:18s} self {} | file {} -> PASS".format(key, _s, _f))
SEAL, DEV, TD_DOC, LR_SEL, FRP = (DOCS["pre_registration"], DOCS["dev_protocol"], DOCS["training_details"],
                                  DOCS["lr_selection"], DOCS["final_run_protocol"])
assert SEAL.get("sealed") is True
assert DEV["parent_seal_self_sha256"] == DOC_SELF["pre_registration"]
assert TD_DOC["parent_seal_self_sha256"] == DOC_SELF["pre_registration"]
assert TD_DOC["parent_dev_protocol_self_sha256"] == DOC_SELF["dev_protocol"]
assert LR_SEL["parent_hashes"] == {"pre_registration": DOC_SELF["pre_registration"],
                                   "dev_protocol": DOC_SELF["dev_protocol"],
                                   "training_details": DOC_SELF["training_details"]}
assert FRP["parent_hashes"] == {k: {"self_sha256": DOC_SELF[k], "file_sha256": DOC_FILE[k]}
                                for k in ("pre_registration", "dev_protocol", "training_details", "lr_selection")}, \
    "final_run_protocol parent_hashes differ from the four documents on disk - aborting."
assert FRP["code_version"] == "v0.6" and FRP["write_once"] is True
print("parent chain: dev_protocol -> seal; training_details -> seal + dev_protocol; lr_selection -> 3; "
      "final_run_protocol.parent_hashes == the four documents (self + file): PASS")

# --- 3. OpenTimestamps proofs: existence, sha256, size (never opened for writing) ---
OTS_FILES = {}
for key, path in DOC_PATHS.items():
    ots = path + ".ots"
    assert os.path.isfile(ots), "OpenTimestamps proof missing: " + ots + " - aborting."
    OTS_FILES[os.path.basename(ots)] = {"sha256": sha256_of(ots), "bytes": os.path.getsize(ots)}
    print("  {}  {:6d} B  {}".format(OTS_FILES[os.path.basename(ots)]["sha256"], OTS_FILES[os.path.basename(ots)]["bytes"],
                                     os.path.basename(ots)))
print(OTS_NOTE)

# --- 4. v0.6 manifest ---
V06_MAN, V06_MAN_SHA = load_verified_manifest(V06_MANIFEST, EXPECTED_V06_MANIFEST_SHA256, "v0.6_manifest.json")
assert V06_MAN["final_run_protocol"]["file_sha256"] == EXPECTED_FILE_SHA256["final_run_protocol"]
assert V06_MAN["final_run_protocol"]["self_sha256"] == EXPECTED_SELF_SHA256["final_run_protocol"]
print("v0.6_manifest.json    : {} | records final_run_protocol file sha256: PASS".format(V06_MAN_SHA))

# --- 5. recipe values loaded FROM dev_protocol.json, then asserted equal to the constants (verbatim v0.5) ---
_rec = DEV["training_recipe"]
EPOCHS = int(_rec["epochs"])
LR_GRID = [float(v) for v in DEV["tuning_grid"]["learning_rate"]]
WEIGHT_DECAY = float(_rec["weight_decay"])
BATCH_SIZE = {k: int(v) for k, v in _rec["batch_size"].items()}
_m = re.search(r"random (\d+)-sample crops", _rec["train_input"])
assert _m, "cannot parse the crop length from dev_protocol train_input"
CROP = int(_m.group(1))
_m = re.search(r"sliding windows of (\d+) samples, stride (\d+) \(window starts \[([0-9, ]+)\]\)", _rec["evaluation"])
assert _m, "cannot parse the evaluation windows from dev_protocol evaluation"
EVAL_WINDOW = int(_m.group(1))
EVAL_STRIDE = int(_m.group(2))
WINDOW_STARTS = [int(v) for v in _m.group(3).split(",")]
AGGREGATION = "mean" if "probability = mean over windows" in _rec["evaluation"] else None
assert "MEAN over sliding windows" in DEV["evaluation_aggregation_note"]
assert _rec["optimizer"] == "AdamW" and _rec["lr_schedule"] == "one-cycle"

assert EPOCHS == EXPECTED_EPOCHS, (EPOCHS, EXPECTED_EPOCHS)
assert LR_GRID == EXPECTED_LR_GRID, (LR_GRID, EXPECTED_LR_GRID)
assert WEIGHT_DECAY == EXPECTED_WEIGHT_DECAY, (WEIGHT_DECAY, EXPECTED_WEIGHT_DECAY)
assert BATCH_SIZE == EXPECTED_BATCH_SIZE, (BATCH_SIZE, EXPECTED_BATCH_SIZE)
assert CROP == EVAL_WINDOW == EXPECTED_CROP, (CROP, EVAL_WINDOW)
assert EVAL_STRIDE == EXPECTED_EVAL_STRIDE and WINDOW_STARTS == EXPECTED_WINDOW_STARTS
assert WINDOW_STARTS == list(range(0, N_SAMPLES - CROP + 1, EVAL_STRIDE))
assert AGGREGATION == "mean"
SELECTION_RULE_TEXT = DEV["selection_rule"]
SANITY_GATE_TEXT = DEV["pipeline_sanity_gate"]
assert "lr is chosen ONLY by the fold-9 macro-AUROC" in SELECTION_RULE_TEXT
assert "macro-AUROC >= 0.90" in SANITY_GATE_TEXT
print("recipe from dev_protocol.json: epochs={} | lr grid={} | weight_decay={} | batch sizes={} | "
      "crop={} | eval windows {} stride {} starts {} | aggregation={} -> all equal the constants: PASS".format(
          EPOCHS, LR_GRID, WEIGHT_DECAY, BATCH_SIZE, CROP, EVAL_WINDOW, EVAL_STRIDE, WINDOW_STARTS, AGGREGATION))

SELECTED_LR = {k: float(v) for k, v in LR_SEL["selected_lr"].items()}
assert SELECTED_LR == EXPECTED_SELECTED_LR, "selected_lr {} != {} - aborting.".format(SELECTED_LR, EXPECTED_SELECTED_LR)
STAGE_PLAN_V07 = FRP["stage_plan"]["v0.7"]
_m = re.search(r"at the locked lr ([0-9.eE+-]+);", STAGE_PLAN_V07)
assert _m and all(float(_m.group(1)) == v for v in SELECTED_LR.values()), \
    "lr in final_run_protocol stage_plan v0.7 != lr_selection.json - aborting."
assert "3 seeds (42, 123, 2024)" in STAGE_PLAN_V07 and "15 new runs" in STAGE_PLAN_V07
assert "(widths 256/259/338)" in STAGE_PLAN_V07
print("selected lr (lr_selection.json) {} == final_run_protocol stage_plan v0.7 lr {}: PASS".format(SELECTED_LR, _m.group(1)))

# --- 6. TRAINING_DETAILS rebuilt VERBATIM from v0.5 and asserted equal to the locked file ---
ADAMW_BETAS = (0.9, 0.99)
ADAMW_EPS = 1e-8
ONECYCLE = {"pct_start": 0.3, "div_factor": 25.0, "final_div_factor": 1e4, "anneal_strategy": "cos",
            "cycle_momentum": True, "base_momentum": 0.85, "max_momentum": 0.95, "three_phase": False}

TRAINING_DETAILS = {
    "optimizer": {"name": "torch.optim.AdamW", "betas": list(ADAMW_BETAS), "eps": ADAMW_EPS,
                  "weight_decay": WEIGHT_DECAY, "weight_decay_source": "dev_protocol.json",
                  "applied_to": "all parameters (incl. BatchNorm and biases)"},
    "lr_scheduler": {"name": "torch.optim.lr_scheduler.OneCycleLR", "max_lr": "the run's lr",
                     "total_steps": "epochs * len(train_loader)", "stepped": "every batch (after optimizer.step())",
                     **ONECYCLE,
                     "momentum_note": ("cycle_momentum/base_momentum/max_momentum are torch defaults: AdamW "
                                       "beta1 is cycled 0.95 -> 0.85 -> 0.95 in the opposite phase of the lr")},
    "epochs": EPOCHS, "epochs_source": "dev_protocol.json",
    "batch_size": BATCH_SIZE, "batch_size_source": "dev_protocol.json",
    "train_data": ("folds 1-8; RandomCropDataset (v0.4): random {}-sample crop per record per epoch, start "
                   "uniform over [0, {}], per-lead normalized with folds 1-8 statistics".format(CROP, N_SAMPLES - CROP)),
    "data_order": ("DataLoader(shuffle=True, generator=torch.Generator().manual_seed(run_seed), num_workers=0, "
                   "pin_memory=True, drop_last=False); RandomCropDataset rng = np.random.default_rng(run_seed)"),
    "seeding": ("set_all_seeds(run_seed) (python random, numpy, torch, torch.cuda) immediately before model "
                "construction; cudnn.deterministic=True, cudnn.benchmark=False"),
    "training_loss": "BCEWithLogitsLoss (mean over batch and the 5 classes)",
    "training_loss_logging": ("per-epoch MEAN TRAINING loss = sample-weighted mean of batch losses over the "
                              "epoch (training data only), saved per run"),
    "validation": ("NO validation metric is computed at any epoch other than the end of epoch {}; no "
                   "checkpoint selection; the model after the last optimizer step is evaluated".format(EPOCHS)),
    "evaluation": ("model.eval(); torch.no_grad(); {} windows (starts {}) of {} samples per fold-9 record, "
                   "per-lead normalized with folds 1-8 statistics; per-window sigmoid PROBABILITIES averaged "
                   "per record (mean of probabilities, not of logits); metadata identical for all windows of a "
                   "record; inference batch size {} (no effect on results)".format(
                       len(WINDOW_STARTS), WINDOW_STARTS, CROP, EVAL_BATCH_SIZE)),
    "metric": ("macro-AUROC = mean over NORM, MI, STTC, CD, HYP of fast_auc (v0.2.1 function); verified once "
               "against sklearn.metrics.roc_auc_score (abs tol {})".format(SKLEARN_CHECK_TOL)),
    "lr_tie_break": ("highest fold-9 macro-AUROC; if two lrs are within {} of each other, choose the SMALLER lr "
                     "(implemented as: among lrs with macro-AUROC >= best - {}, take the smallest)".format(
                         LR_TIE_TOL, LR_TIE_TOL)),
    "per_run_outputs": ("v0.5_{arch}_{arm}_seed{seed}_lr{lr}_fold9_probs.csv (ecg_id, patient_id, 5 "
                        "probabilities), _train_loss.csv (per-epoch mean training loss), _state_dict.pt "
                        "(final model); all hashed in the manifest"),
    "determinism_note": ("torch.use_deterministic_algorithms is NOT enabled; some CUDA kernels may not be "
                         "bitwise reproducible across re-runs"),
}
assert plain(TRAINING_DETAILS) == TD_DOC["details"], \
    "v0.5_training_details.json on disk differs from this code's training definition - aborting."
TRAINING_DETAILS_SHA = DOC_SELF["training_details"]
print("TRAINING_DETAILS (this code) == v0.5_training_details.json['details']: PASS "
      "(per_run_outputs text keeps the v0.5 prefix verbatim; v0.7 files use the v0.7 prefix)")

# --- 7. input CSVs, v0.4 manifest, signal cache, lead-norm csv ---
DB_CSV = os.path.join(DATA_ROOT, "ptbxl_database.csv")
SCP_CSV = os.path.join(DATA_ROOT, "scp_statements.csv")
for p in (DB_CSV, SCP_CSV):
    assert os.path.isfile(p), "file not found: " + p
INPUT_HASHES = {os.path.basename(p): sha256_of(p) for p in (DB_CSV, SCP_CSV)}
assert INPUT_HASHES == EXPECTED_INPUT_HASHES == FRP["input_files_sha256"], "input CSV hashes differ - aborting."
print("input CSV hashes      : equal to v0.4-v0.6 and to final_run_protocol -> PASS")
V04_MAN, V04_MAN_SHA = load_verified_manifest(V04_MANIFEST, EXPECTED_V04_MANIFEST_SHA256, "v0.4_manifest.json")
CACHE_HASHES = {}
for path, expected in [(SIGNALS_NPY, EXPECTED_SIGNALS_NPY_SHA256), (SIGNALS_INDEX, EXPECTED_SIGNALS_INDEX_SHA256),
                       (V04_LEAD_NORM_CSV, EXPECTED_LEAD_NORM_CSV_SHA256)]:
    assert os.path.isfile(path), "v0.4 file missing: " + path
    got = sha256_of(path)
    assert got == expected == manifest_entry(V04_MAN, os.path.basename(path))["sha256"] \
        == FRP["v0.4_signal_cache_sha256"][os.path.basename(path)], "v0.4 file hash mismatch for " + path + " - aborting."
    CACHE_HASHES[os.path.basename(path)] = got
    print("  {}  {}  -> PASS (== v0.4 manifest == final_run_protocol)".format(got, os.path.basename(path)))

# --- 8. v0.5 manifest and the 9 reused run files ---
V05_MAN, V05_MAN_SHA = load_verified_manifest(V05_MANIFEST, EXPECTED_V05_MANIFEST_SHA256, "v0.5_manifest.json")
assert FRP["earlier_manifests_sha256"]["v0.5_manifest.json"] == V05_MAN_SHA


def v05_run_paths(arch, arm, seed, lr):
    base = os.path.join(V05_DIR, "v0.5_{}_{}_seed{}_lr{:g}".format(arch, arm, seed, lr))
    return {"probs": base + "_fold9_probs.csv", "loss": base + "_train_loss.csv", "state": base + "_state_dict.pt"}


REUSED_FILES = {}
for arch, arm, seed in REUSED_CELLS:
    for kind, p in v05_run_paths(arch, arm, seed, SELECTED_LR[arch]).items():
        name = os.path.basename(p)
        assert name in FRP["v0.7_reused_v0.5_files"], name + " is not listed in final_run_protocol - aborting."
        assert os.path.isfile(p), "reused v0.5 file missing: " + p
        got = sha256_of(p)
        assert got == FRP["v0.7_reused_v0.5_files"][name] == manifest_entry(V05_MAN, name)["sha256"], \
            "reused file hash mismatch (disk / protocol / v0.5 manifest) for " + name + " - aborting."
        REUSED_FILES[name] = {"path": p, "sha256": got}
assert set(REUSED_FILES) == set(FRP["v0.7_reused_v0.5_files"]), "reused-file set differs from the protocol - aborting."
print("v0.5_manifest.json    : {} | {} reused files: disk == final_run_protocol == v0.5 manifest -> PASS".format(
    V05_MAN_SHA, len(REUSED_FILES)))

# --- 9. reference implementation, read FROM the protocol file (never retyped) ---
REFERENCE_CODE = FRP["reference_implementation"]["code"]
REFERENCE_CODE_SHA256 = hashlib.sha256(REFERENCE_CODE.encode("utf-8")).hexdigest()
assert REFERENCE_CODE_SHA256 == FRP["reference_implementation"]["sha256_of_utf8_code"] == EXPECTED_REFERENCE_CODE_SHA256, \
    "reference implementation sha256 mismatch - aborting."
REF = {}
exec(REFERENCE_CODE, REF)
print("reference implementation: sha256 {} -> PASS (exec'd into REF; every v0.7 bootstrap number uses REF)".format(
    REFERENCE_CODE_SHA256))

# --- existing v0.7 artefacts (re-execution) ---
PLAN_PATH = os.path.join(OUT_DIR, PREFIX + "exploratory_analysis_plan.json")
ATTEMPT_LOG_PATH = os.path.join(OUT_DIR, PREFIX + "attempt_log.jsonl")
PLAN_EXISTS = os.path.exists(PLAN_PATH)
PLAN_ON_DISK = None
if PLAN_EXISTS:
    _s, _r, PLAN_ON_DISK = verify_self_sha256(PLAN_PATH)
    assert _s == _r, "v0.7_exploratory_analysis_plan.json BROKEN (stored != recomputed self_sha256) - aborting."
    print("exploratory analysis plan EXISTS (self_sha256 {}, verified) -> compared field by field in Part C.".format(_s))
print("attempt log: {}".format("present (re-execution)" if os.path.exists(ATTEMPT_LOG_PATH) else "not present"))

os.makedirs(OUT_DIR, exist_ok=True)
# =====================================================================================
# DATA (CSV metadata, folds 1-9 only), LABELS, METADATA PREPROCESSING
# =====================================================================================
hdr("DATA, LABELS, METADATA")

df_all = pd.read_csv(DB_CSV, index_col="ecg_id")
df9 = df_all[df_all["strat_fold"] <= 9].copy()
mask_train = df9["strat_fold"] <= 8
mask_eval = df9["strat_fold"] == 9
assert (df9["strat_fold"] == 10).sum() == 0, "fold 10 leaked into the working set - aborting."
n_fold10_in_source = int((df_all["strat_fold"] == 10).sum())
df_all = df_all[df_all["strat_fold"] <= 9]      # v0.7 change (5): no fold-10 value stays in memory
gc.collect()
train_patients = set(df9.loc[mask_train, "patient_id"])
eval_patients = set(df9.loc[mask_eval, "patient_id"])
assert train_patients.isdisjoint(eval_patients), \
    "patient-level leakage between train (folds 1-8) and eval (fold 9) - aborting."
current_index = pd.DataFrame({
    "ecg_id": df9.index.to_numpy(),
    "strat_fold": df9["strat_fold"].to_numpy(),
    "patient_id": df9["patient_id"].to_numpy(),
})
N_REC = len(current_index)
assert N_REC == EXPECTED_N_RECORDS_FOLDS1TO9
train_rows = np.where(mask_train.to_numpy())[0]
eval_rows = np.where(mask_eval.to_numpy())[0]
print("records: folds 1-8 = {} | fold 9 = {} | fold-10 records in source CSV, never opened: {}".format(
    len(train_rows), len(eval_rows), n_fold10_in_source))

# --- label derivation (verbatim from v0.2.1) ---
scp = pd.read_csv(SCP_CSV, index_col=0)
diag = scp[scp["diagnostic"].fillna(0) == 1]
code2class = {k: v for k, v in diag["diagnostic_class"].items() if isinstance(v, str)}
parsed = df9["scp_codes"].apply(ast.literal_eval)
rec_classes = parsed.apply(lambda d: {code2class[k] for k in d if k in code2class})
lab = pd.DataFrame({c: rec_classes.apply(lambda s, c=c: int(c in s)) for c in SUPERCLASSES},
                   index=df9.index)
assert (current_index["ecg_id"].to_numpy() == lab.index.to_numpy()).all()
fold9_pos = {c: int(lab.loc[mask_eval, c].sum()) for c in SUPERCLASSES}
assert fold9_pos == EXPECTED_FOLD9_POSITIVES, "fold-9 label counts differ from v0.2.1 - aborting."
print("fold-9 positives:", fold9_pos, "-> PASS")
Y = lab[SUPERCLASSES].to_numpy(dtype=np.float32)
Y_EVAL = {sc: lab.loc[mask_eval, sc].to_numpy() for sc in SUPERCLASSES}

# --- metadata matrices (verbatim from v0.4 Part C) ---
age_raw = df9["age"]
age_sentinel_mask = (age_raw == 300)
n_age_sentinel = int(age_sentinel_mask.sum())
age_clean = age_raw.where(~age_sentinel_mask, np.nan)
age_censored = age_sentinel_mask.astype(float)
age_imputed = age_clean.fillna(90.0)
sex = df9["sex"].astype(float)
print("age: sentinel 300 affects", n_age_sentinel, "records; flagged via age_censored, imputed with 90.")

device_level = to_level_string(df9["device"])
site_level = to_level_string(df9["site"])
nurse_level = to_level_string(df9["nurse"])
device_dummies, device_categories = encode_categorical(device_level, mask_train, "device")
site_dummies, site_categories = encode_categorical(site_level, mask_train, "site")
nurse_dummies, nurse_categories = encode_categorical(nurse_level, mask_train, "nurse")
for name, dmy in [("device", device_dummies), ("site", site_dummies), ("nurse", nurse_dummies)]:
    assert dmy.shape[1] == EXPECTED_ONEHOT_COUNTS[name], \
        "{} one-hot width {} != v0.2.1's {}".format(name, dmy.shape[1], EXPECTED_ONEHOT_COUNTS[name])

age_scaler = StandardScaler().fit(age_imputed[mask_train].to_numpy().reshape(-1, 1))
age_imputed_std = pd.Series(age_scaler.transform(age_imputed.to_numpy().reshape(-1, 1)).ravel(),
                            index=df9.index, name="age_imputed")

B1_cols = ["age_imputed", "age_censored", "sex"]
B2_cols = list(device_dummies.columns) + list(site_dummies.columns) + list(nurse_dummies.columns)
B3_cols = B1_cols + B2_cols

META_FEATURES = pd.concat([age_imputed_std, age_censored.rename("age_censored"), sex.rename("sex"),
                           device_dummies, site_dummies, nurse_dummies], axis=1)
ARM_COLUMNS = {"signal_only": [], "signal_plus_physio": B1_cols, "signal_plus_all": B3_cols}
META = {}
for arm in ARMS:
    cols = ARM_COLUMNS[arm]
    if cols:
        META[arm] = META_FEATURES[cols].to_numpy(dtype=np.float32)
    else:
        META[arm] = np.zeros((N_REC, 0), dtype=np.float32)
    assert META[arm].shape == (N_REC, EXPECTED_META_DIM[arm]), \
        "{}: metadata dim {} != {}".format(arm, META[arm].shape[1], EXPECTED_META_DIM[arm])
    assert np.isfinite(META[arm]).all()
    print("  {:20s} metadata dim = {:3d}{}".format(arm, META[arm].shape[1],
          "" if not cols else "  ({} ... {})".format(", ".join(cols[:3]), cols[-1])))
assert ARM_COLUMNS["signal_plus_physio"] == B1_cols and ARM_COLUMNS["signal_plus_all"] == B1_cols + B2_cols
print("column order: physio == v0.2.1 B1_physio, all == v0.2.1 B3_physio_plus_acquisition: PASS")
print("age_imputed scaler (folds 1-8): mean={:.4f}, scale={:.4f}".format(
    float(age_scaler.mean_[0]), float(age_scaler.scale_[0])))
assert abs(float(age_scaler.mean_[0]) - V05_AGE_SCALER_FOLDS1TO8["mean"]) <= AGE_SCALER_TOL and \
    abs(float(age_scaler.scale_[0]) - V05_AGE_SCALER_FOLDS1TO8["scale"]) <= AGE_SCALER_TOL, \
    "folds 1-8 age scaler differs from v0.5 - aborting."
assert FRP["locked_quantities"]["meta_dim_development_folds1to8"] == EXPECTED_META_DIM, \
    "final_run_protocol meta_dim_development_folds1to8 != {} - aborting.".format(EXPECTED_META_DIM)
print("age scaler == v0.5 (tol {}) and meta dims == final_run_protocol meta_dim_development_folds1to8: PASS".format(
    AGE_SCALER_TOL))


# =====================================================================================
# PART A - ENVIRONMENT REGRESSION CHECK (CPU)
# =====================================================================================
hdr("PART A - ENVIRONMENT REGRESSION CHECK (v0.3 LogisticRegression, CPU)")

import torch
VERSIONS = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
            "scipy": scipy.__version__, "sklearn": sklearn.__version__, "torch": torch.__version__,
            "torch_cuda": torch.version.cuda, "os": platform.platform()}
print("versions:", VERSIONS)

# v0.2.1 feature construction and LogisticRegression setup, verbatim, for B1 / B3 only
FEATURES_V021 = pd.concat([age_imputed.rename("age_imputed"), age_censored.rename("age_censored"),
                           sex.rename("sex"), device_dummies, site_dummies, nurse_dummies], axis=1)
y_train_full = lab.loc[mask_train]
regression_rows = []
for block_name, cols in [("B1_physio", B1_cols), ("B3_physio_plus_acquisition", B3_cols)]:
    Xtr_raw = FEATURES_V021.loc[mask_train, cols].to_numpy(dtype=float)
    Xev_raw = FEATURES_V021.loc[mask_eval, cols].to_numpy(dtype=float)
    block_cols = cols
    continuous_cols = [c for c in block_cols if c in ("age_imputed", "recording_year")]
    cont_idx = [block_cols.index(c) for c in continuous_cols]
    rest_idx = [i for i in range(len(block_cols)) if i not in cont_idx]
    scaler = StandardScaler().fit(Xtr_raw[:, cont_idx])
    Xtr_lr = np.empty_like(Xtr_raw)
    Xev_lr = np.empty_like(Xev_raw)
    Xtr_lr[:, cont_idx] = scaler.transform(Xtr_raw[:, cont_idx])
    Xev_lr[:, cont_idx] = scaler.transform(Xev_raw[:, cont_idx])
    if rest_idx:
        Xtr_lr[:, rest_idx] = Xtr_raw[:, rest_idx]
        Xev_lr[:, rest_idx] = Xev_raw[:, rest_idx]
    per_class = []
    for sc in SUPERCLASSES:
        model_lr = LogisticRegression(max_iter=2000, C=1.0, solver="lbfgs", random_state=SEED)
        model_lr.fit(Xtr_lr, y_train_full[sc].to_numpy())
        pos_col = list(model_lr.classes_).index(1)
        per_class.append(fast_auc(Y_EVAL[sc], model_lr.predict_proba(Xev_lr)[:, pos_col]))
    macro = float(np.nanmean(per_class))
    ref = V03_LR_MACRO_AUROC[block_name]
    ok = abs(macro - ref) <= ENV_REGRESSION_TOL
    regression_rows.append({"block": block_name, "learner": "LogisticRegression", "seed": SEED,
                            "macro_auroc_now": macro, "macro_auroc_v0.3": ref, "abs_diff": abs(macro - ref),
                            "tolerance": ENV_REGRESSION_TOL, "pass": ok, **VERSIONS})
    print("  {:28s} now={:.6f}  v0.3={:.6f}  |diff|={:.2e}  -> {}".format(
        block_name, macro, ref, abs(macro - ref), "PASS" if ok else "FAIL"))
env_reg_df = pd.DataFrame(regression_rows)
save_csv(env_reg_df, PREFIX + "env_regression.csv", index=False)
assert all(r["pass"] for r in regression_rows), \
    "ENVIRONMENT REGRESSION FAILED: v0.3 LogisticRegression results do not reproduce - aborting."
print("environment regression check: PASS")


# =====================================================================================
# GPU SETUP, SIGNAL CACHE (v0.4), LEAD NORMALIZATION
# =====================================================================================
hdr("GPU SETUP AND SIGNAL CACHE")

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

if not torch.cuda.is_available():
    raise RuntimeError("torch.cuda.is_available() is False - no usable CUDA GPU in this kernel. Aborting.")
DEVICE = torch.device("cuda:0")


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_all_seeds(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
GPU_NAME = torch.cuda.get_device_name(0)
print("gpu: {} | torch {} | CUDA {} | cuDNN {} | cudnn.deterministic=True, benchmark=False".format(
    GPU_NAME, torch.__version__, torch.version.cuda, torch.backends.cudnn.version()))

for path, expected in [(SIGNALS_NPY, EXPECTED_SIGNALS_NPY_SHA256), (SIGNALS_INDEX, EXPECTED_SIGNALS_INDEX_SHA256)]:
    assert os.path.isfile(path), "v0.4 cache file missing: " + path
    got = sha256_of(path)
    print("  {}  {}  -> {}".format(got, os.path.basename(path), "PASS" if got == expected else "FAIL"))
    assert got == expected, "v0.4 cache hash mismatch for " + path + " - aborting."
_old_idx = pd.read_csv(SIGNALS_INDEX, encoding="utf-8-sig")
assert list(_old_idx.columns) == list(current_index.columns) and _old_idx.astype(str).reset_index(drop=True).equals(
    current_index.astype(str).reset_index(drop=True)), "v0.4 sidecar index does not match folds 1-9 - aborting."
X = np.load(SIGNALS_NPY)
assert X.shape == (N_REC, N_LEADS, N_SAMPLES) and X.dtype == np.float32
print("signals loaded from the v0.4 cache: {} {} (folds 1-9 only)".format(X.shape, X.dtype))

# --- per-lead normalization parameters: folds 1-8 ONLY (two-pass, float64; verbatim v0.4) ---
CHUNK = 1000
_sum = np.zeros(N_LEADS, dtype=np.float64)
for s in range(0, len(train_rows), CHUNK):
    _sum += X[train_rows[s:s + CHUNK]].astype(np.float64).sum(axis=(0, 2))
_n_vals = len(train_rows) * N_SAMPLES
LEAD_MEAN = _sum / _n_vals
_sq = np.zeros(N_LEADS, dtype=np.float64)
for s in range(0, len(train_rows), CHUNK):
    xc = X[train_rows[s:s + CHUNK]].astype(np.float64) - LEAD_MEAN[None, :, None]
    _sq += (xc ** 2).sum(axis=(0, 2))
LEAD_STD = np.sqrt(_sq / _n_vals)
assert np.all(LEAD_STD > 0)
_v04_norm = pd.read_csv(V04_LEAD_NORM_CSV, encoding="utf-8-sig")
assert np.allclose(_v04_norm["mean"].to_numpy(), LEAD_MEAN, rtol=0, atol=1e-12) and \
    np.allclose(_v04_norm["std"].to_numpy(), LEAD_STD, rtol=0, atol=1e-12), \
    "lead normalization differs from v0.4_lead_norm_params.csv - aborting."
print("per-lead normalization (folds 1-8) recomputed and equal to v0.4_lead_norm_params.csv: PASS")
LEAD_MEAN_F32 = LEAD_MEAN.astype(np.float32)[:, None]
LEAD_STD_F32 = LEAD_STD.astype(np.float32)[:, None]


# =====================================================================================
# MODELS (verbatim from v0.4) + fusion check against dev_protocol.json
# =====================================================================================
# >>> MODELS
# SOURCE FOLLOWED: Strodthoff, Wagner, Schaeffter, Samek (2021) "Deep Learning for ECG
# Analysis: Benchmarks and Insights from PTB-XL", reference code repository
# helme/ecg_ptbxl_benchmarking: code/models/resnet1d.py (resnet1d_wang, BasicBlock1d,
# ResNet1d), code/models/inception1d.py (Inception1d, InceptionBlock1d, Shortcut1d),
# code/models/basic_conv1d.py (create_head1d, bn_drop_lin, AdaptiveConcatPool1d), and
# code/models/fastai_model.py (how the benchmark instantiates them). Re-implemented here
# from memory of that code in plain PyTorch (no fastai); the repository was NOT
# consulted at write time.
#
# DETAILS I AM UNSURE OF (should be checked against the repository):
#  1) resnet1d_wang kernel sizes: factory default kernel_size=[5,3], stem kernel 7,
#     stride_stem=1, no stem pooling, inplanes=128, layers [1,1,1], fix_feature_dim=True
#     (all stages 128 channels), stride 2 in stages 2 and 3. If fastai_model.py passes
#     the int kernel_size=5, BasicBlock1d expands it to [5, 5//2+1] = [5,3] -> same.
#  2) inception1d as used by the benchmark: kernel_size = 8*5 = 40 -> branch kernels
#     [39,19,9], depth 6, bottleneck 32, nb_filters 32 (output 4*32 = 128 channels),
#     residual shortcut every 3 blocks; all convs bias=False.
#  3) Head (create_head1d) with lin_ftrs_head=[128], ps_head=0.5, bn=True, relu,
#     concat pooling: AdaptiveConcatPool1d [max, avg] -> 2*nf -> BN, Dropout(0.25),
#     Linear(2*nf,128), ReLU -> BN, Dropout(0.5), Linear(128, 5). I believe the
#     benchmark default is lin_ftrs_head=[128]; with lin_ftrs_head=None the head would be
#     a single Linear(2*nf, 5) instead.
#  4) Order inside AdaptiveConcatPool1d is [max, avg] (fastai convention).
#  5) The reference aggregates sliding-window predictions with aggregate_fn="max" by
#     default in fastai_model.py (I am fairly but not fully sure); the project's dev
#     protocol uses the MEAN over windows (Part F). Flagged, not changed.
#  6) MaxPool1d(3, stride, padding=1) on the inception max-pool branch.
#
# METADATA FUSION (v0.4 PATCH FIX 1 -- sealed text is binding: "concatenated to the
# network's global-pooled embedding before the final linear classification head").
# The global-pooled embedding is the 2*nf = 256-dim AdaptiveConcatPool1d output; the
# "head" is the whole MLP head. The head's first block keeps the reference order
# (BN, Dropout(0.25), Linear, ReLU), split as: BN + Dropout applied to the embedding
# (`pool_norm`, identical across arms), then the RAW metadata (no BN, no dropout) is
# concatenated, then Linear(256 + meta_dim, 128) -> ReLU -> BN -> Dropout(0.5) ->
# Linear(128, 5). Backbone, pooling and pool_norm are identical across arms; the head's
# first Linear differs in input width by design.


def conv1d_same(in_planes, out_planes, kernel_size=3, stride=1):
    return nn.Conv1d(in_planes, out_planes, kernel_size=kernel_size, stride=stride,
                     padding=(kernel_size - 1) // 2, bias=False)


class BasicBlock1d(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, kernel_size=(3, 3), downsample=None):
        super().__init__()
        if isinstance(kernel_size, int):
            kernel_size = [kernel_size, kernel_size // 2 + 1]
        self.conv1 = conv1d_same(inplanes, planes, kernel_size=kernel_size[0], stride=stride)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv1d_same(planes, planes, kernel_size=kernel_size[1])
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample

    def forward(self, x):
        residual = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + residual)


def resnet1d_wang_backbone(input_channels=12, inplanes=128, kernel_size=(5, 3), kernel_size_stem=7,
                           stride_stem=1, layers=(1, 1, 1), stride=2):
    mods = [nn.Conv1d(input_channels, inplanes, kernel_size=kernel_size_stem, stride=stride_stem,
                      padding=(kernel_size_stem - 1) // 2, bias=False),
            nn.BatchNorm1d(inplanes), nn.ReLU(inplace=True)]          # no stem pooling
    cur = inplanes
    for i, n_blocks in enumerate(layers):
        st = 1 if i == 0 else stride
        planes = inplanes                                             # fix_feature_dim=True
        downsample = None
        if st != 1 or cur != planes:
            downsample = nn.Sequential(nn.Conv1d(cur, planes, kernel_size=1, stride=st, bias=False),
                                       nn.BatchNorm1d(planes))
        blocks = [BasicBlock1d(cur, planes, st, list(kernel_size), downsample)]
        cur = planes
        for _ in range(1, n_blocks):
            blocks.append(BasicBlock1d(cur, planes))
        mods.append(nn.Sequential(*blocks))
    return nn.Sequential(*mods), cur


class InceptionBlock1d(nn.Module):
    def __init__(self, ni, nb_filters, kss, stride=1, bottleneck_size=32):
        super().__init__()
        self.bottleneck = conv1d_same(ni, bottleneck_size, 1, stride) if bottleneck_size > 0 else nn.Identity()
        bi = bottleneck_size if bottleneck_size > 0 else ni
        self.convs = nn.ModuleList([conv1d_same(bi, nb_filters, ks) for ks in kss])
        self.conv_bottle = nn.Sequential(nn.MaxPool1d(3, stride, padding=1), conv1d_same(ni, nb_filters, 1))
        self.bn_relu = nn.Sequential(nn.BatchNorm1d((len(kss) + 1) * nb_filters), nn.ReLU())

    def forward(self, x):
        b = self.bottleneck(x)
        return self.bn_relu(torch.cat([c(b) for c in self.convs] + [self.conv_bottle(x)], dim=1))


class Shortcut1d(nn.Module):
    def __init__(self, ni, nf):
        super().__init__()
        self.act_fn = nn.ReLU(True)
        self.conv = conv1d_same(ni, nf, 1)
        self.bn = nn.BatchNorm1d(nf)

    def forward(self, inp, out):
        return self.act_fn(out + self.bn(self.conv(inp)))


class InceptionBackbone(nn.Module):
    def __init__(self, input_channels=12, kernel_size=40, depth=6, bottleneck_size=32, nb_filters=32,
                 use_residual=True):
        super().__init__()
        assert kernel_size >= 40 and depth % 3 == 0
        kss = [k - 1 if k % 2 == 0 else k for k in [kernel_size, kernel_size // 2, kernel_size // 4]]
        n_ks = len(kss) + 1
        self.depth, self.use_residual = depth, use_residual
        self.im = nn.ModuleList([InceptionBlock1d(input_channels if d == 0 else n_ks * nb_filters,
                                                  nb_filters, kss, bottleneck_size=bottleneck_size)
                                 for d in range(depth)])
        self.sk = nn.ModuleList([Shortcut1d(input_channels if d == 0 else n_ks * nb_filters, n_ks * nb_filters)
                                 for d in range(depth // 3)])
        self.out_channels = n_ks * nb_filters
        self.kss = kss

    def forward(self, x):
        input_res = x
        for d in range(self.depth):
            x = self.im[d](x)
            if self.use_residual and d % 3 == 2:
                x = self.sk[d // 3](input_res, x)
                input_res = x
        return x


class AdaptiveConcatPool1d(nn.Module):
    def __init__(self):
        super().__init__()
        self.mp = nn.AdaptiveMaxPool1d(1)
        self.ap = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        return torch.cat([self.mp(x), self.ap(x)], dim=1).flatten(1)


class ECGNet(nn.Module):
    """Backbone + reference head. Metadata (meta_dim >= 0) is concatenated, raw, to the
    global-pooled embedding (after its BN + Dropout) and enters the head's first Linear.
    backbone / pool / pool_norm are built in the same order with the same shapes for
    every arm, so with the same seed they are initialised identically (checked below)."""

    def __init__(self, arch, meta_dim, n_classes=5, input_channels=12, lin_ftrs=128, ps_head=0.5):
        super().__init__()
        if arch == "resnet1d_wang":
            self.backbone, nf = resnet1d_wang_backbone(input_channels=input_channels)
        elif arch == "inception1d":
            self.backbone = InceptionBackbone(input_channels=input_channels, kernel_size=40)
            nf = self.backbone.out_channels
        else:
            raise ValueError(arch)
        self.pool = AdaptiveConcatPool1d()
        # embedding pre-processing: identical across arms
        self.pool_norm = nn.Sequential(nn.BatchNorm1d(2 * nf), nn.Dropout(ps_head / 2))
        self.embedding_dim = 2 * nf
        self.meta_dim = meta_dim
        # head: receives [embedding (+ raw metadata)]
        self.hidden = nn.Sequential(nn.Linear(2 * nf + meta_dim, lin_ftrs), nn.ReLU(inplace=True),
                                    nn.BatchNorm1d(lin_ftrs), nn.Dropout(ps_head))
        self.final = nn.Linear(lin_ftrs, n_classes)

    def forward(self, x, meta=None):
        z = self.pool_norm(self.pool(self.backbone(x)))
        if self.meta_dim > 0:
            z = torch.cat([z, meta], dim=1)   # raw metadata: no BN, no dropout
        return self.final(self.hidden(z))
# <<< MODELS


METADATA_FUSION_DESCRIPTION = (
    "location: the metadata vector is concatenated to the network's global-pooled embedding (the "
    "2*nf = 256-dim AdaptiveConcatPool1d [max, avg] output, after that embedding's BatchNorm1d + "
    "Dropout(0.25)) and the concatenation is the input of the head's first Linear "
    "(256 + meta_dim -> 128), followed by ReLU, BatchNorm1d(128), Dropout(0.5), Linear(128 -> 5); "
    "metadata enters RAW (age_imputed standardized with training-fold statistics, all other columns "
    "0/1): no BatchNorm and no dropout is applied to the metadata; identical across arms: backbone, "
    "global pooling and the embedding's BatchNorm + Dropout (same seed -> identical initial weights, "
    "asserted in v0.4); differs across arms by design: only the input width of the head's first "
    "Linear (256 / 259 / 338 for signal_only / signal_plus_physio / signal_plus_all)")
assert METADATA_FUSION_DESCRIPTION == DEV["metadata_fusion"], \
    "ECGNet fusion description differs from the one locked in dev_protocol.json - aborting."
print("ECGNet metadata fusion == dev_protocol.json 'metadata_fusion': PASS")


class RandomCropDataset(Dataset):
    """Training records -> random 250-sample crop, per-lead normalized with folds 1-8 stats."""

    def __init__(self, X, rows, meta, y, crop, seed):
        self.X, self.rows, self.meta, self.y, self.crop = X, rows, meta, y, crop
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        s = int(self.rng.integers(0, N_SAMPLES - self.crop + 1))
        x = (self.X[r, :, s:s + self.crop] - LEAD_MEAN_F32) / LEAD_STD_F32
        return (torch.from_numpy(np.ascontiguousarray(x, dtype=np.float32)),
                torch.from_numpy(self.meta[r]), torch.from_numpy(self.y[r]))



# =====================================================================================
# RUN HELPERS (verbatim v0.5 naming / prediction / loading; saving + training per changes 1-4)
# =====================================================================================
def run_name(arch, arm, seed, lr):
    return "{}{}_{}_seed{}_lr{:g}".format(PREFIX, arch, arm, seed, lr)


def run_paths(arch, arm, seed, lr):
    base = os.path.join(OUT_DIR, run_name(arch, arm, seed, lr))
    return {"probs": base + "_fold9_probs.csv", "loss": base + "_train_loss.csv", "state": base + "_state_dict.pt"}


_SKLEARN_VERIFIED = {"done": False}


def macro_auroc_from_probs(probs):
    """probs: (n_fold9, 5) in eval_rows order. Returns (macro, per-class dict)."""
    per = {sc: fast_auc(Y_EVAL[sc], probs[:, k]) for k, sc in enumerate(SUPERCLASSES)}
    if not _SKLEARN_VERIFIED["done"]:
        for k, sc in enumerate(SUPERCLASSES):
            sk = float(roc_auc_score(Y_EVAL[sc], probs[:, k]))
            assert abs(sk - per[sc]) <= SKLEARN_CHECK_TOL, "fast_auc != sklearn roc_auc_score for " + sc
        _SKLEARN_VERIFIED["done"] = True
        print("    fast_auc verified against sklearn.metrics.roc_auc_score (abs tol {}): PASS".format(SKLEARN_CHECK_TOL))
    return float(np.mean([per[sc] for sc in SUPERCLASSES])), per


@torch.no_grad()
def predict_fold9(model, arm):
    """Mean over the 7 windows of per-window sigmoid probabilities, per fold-9 record."""
    model.eval()
    probs = np.zeros((len(eval_rows), len(SUPERCLASSES)), dtype=np.float64)
    meta_all = META[arm]
    for st in WINDOW_STARTS:
        for b in range(0, len(eval_rows), EVAL_BATCH_SIZE):
            rows = eval_rows[b:b + EVAL_BATCH_SIZE]
            xw = (X[rows, :, st:st + CROP] - LEAD_MEAN_F32) / LEAD_STD_F32
            xb = torch.from_numpy(np.ascontiguousarray(xw, dtype=np.float32)).to(DEVICE)
            mb = torch.from_numpy(meta_all[rows]).to(DEVICE)
            probs[b:b + len(rows)] += torch.sigmoid(model(xb, mb)).double().cpu().numpy()
    return probs / len(WINDOW_STARTS)


def load_saved_probs(paths):
    pdf = pd.read_csv(paths["probs"], encoding="utf-8-sig")
    assert (pdf["ecg_id"].to_numpy() == current_index["ecg_id"].to_numpy()[eval_rows]).all()
    return pdf[["prob_" + sc for sc in SUPERCLASSES]].to_numpy(dtype=np.float64)


def _partial(p):
    return p + ".partial"


def save_run_outputs(paths, probs, loss_rows, model):
    """v0.7 change (3): state_dict and loss CSV first, the probability CSV LAST; every file is written
    to '<name>.partial' and renamed, so an existing probability CSV implies three complete files."""
    torch.save(model.state_dict(), _partial(paths["state"]))
    os.replace(_partial(paths["state"]), paths["state"])
    pd.DataFrame(loss_rows).to_csv(_partial(paths["loss"]), index=False, encoding="utf-8-sig")
    os.replace(_partial(paths["loss"]), paths["loss"])
    pdf = pd.DataFrame({"ecg_id": current_index["ecg_id"].to_numpy()[eval_rows],
                        "patient_id": current_index["patient_id"].to_numpy()[eval_rows]})
    for k, sc in enumerate(SUPERCLASSES):
        pdf["prob_" + sc] = probs[:, k]
    pdf.to_csv(_partial(paths["probs"]), index=False, encoding="utf-8-sig")
    os.replace(_partial(paths["probs"]), paths["probs"])
    return {kind: sha256_of(p) for kind, p in paths.items()}


# --- append-only attempt log ---
def read_attempt_log():
    entries = []
    if not os.path.exists(ATTEMPT_LOG_PATH):
        return entries
    with open(ATTEMPT_LOG_PATH, "r", encoding="utf-8") as f:
        raw = f.read()
    for i, line in enumerate(raw.splitlines()):
        if not line.strip():
            continue
        try:
            entries.append(json.loads(line))
        except json.JSONDecodeError:
            warn("attempt log line {} is not valid JSON (partial write?) - ignored: {!r}".format(i + 1, line[:120]))
    if raw and not raw.endswith("\n"):
        with open(ATTEMPT_LOG_PATH, "a", encoding="utf-8") as f:   # terminate a partial last line (append-only)
            f.write("\n")
    return entries


def log_event(run, attempt, event, **fields):
    entry = {"run": run, "attempt": attempt, "event": event, "time": datetime.now().astimezone().isoformat(), **fields}
    with open(ATTEMPT_LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(plain(entry), ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())
    ATTEMPT_LOG.append(plain(entry))
    return entry


def train_run(arch, arm, seed, lr, attempt):
    """Train 50 epochs on folds 1-8, evaluate ONCE on fold 9 at the end, save run files.
    v0.7 change (1): the v0.5 FORBIDDEN_ARM guard ("signal_plus_all must not be trained in v0.5") was
    REMOVED deliberately: final_run_protocol.json stage_plan["v0.7"] trains all three arms."""
    assert TRAINING_DETAILS_SHA is not None and os.path.isfile(TRAINING_DETAILS_PATH), \
        "training details must be written before any training run"
    assert os.path.isfile(PLAN_PATH), "the exploratory analysis plan must be written before any training run"
    name = run_name(arch, arm, seed, lr)
    paths = run_paths(arch, arm, seed, lr)
    bs = BATCH_SIZE[arch]
    assert len(train_rows) % bs != 1, "last training batch would have size 1 (BatchNorm) - aborting."
    set_all_seeds(seed)
    model = ECGNet(arch, META[arm].shape[1]).to(DEVICE)
    assert model.hidden[0].in_features == EXPECTED_HEAD_IN[arm], \
        "{}: head first-Linear in_features {} != {}".format(name, model.hidden[0].in_features, EXPECTED_HEAD_IN[arm])
    ds = RandomCropDataset(X, train_rows, META[arm], Y, CROP, seed)
    loader = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
                        drop_last=False, generator=torch.Generator().manual_seed(seed))
    assert EPOCHS * len(loader) == EXPECTED_TOTAL_STEPS_DEV, \
        "OneCycle total_steps {} != {}".format(EPOCHS * len(loader), EXPECTED_TOTAL_STEPS_DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=ADAMW_BETAS, eps=ADAMW_EPS, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=EPOCHS * len(loader), **ONECYCLE)
    crit = nn.BCEWithLogitsLoss()
    loss_rows = []
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum = torch.zeros((), device=DEVICE, dtype=torch.float64)
        n_seen = 0
        for xb, mb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            mb = mb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(xb, mb), yb)
            loss.backward()
            opt.step()
            sched.step()
            loss_sum += loss.detach().double() * xb.shape[0]
            n_seen += xb.shape[0]
        mean_loss = float(loss_sum.item()) / n_seen
        assert math.isfinite(mean_loss), "{}: non-finite training loss at epoch {} - aborting.".format(name, epoch)
        loss_rows.append({"epoch": epoch, "mean_train_loss": mean_loss, "lr_end_of_epoch": sched.get_last_lr()[0]})
        if epoch % 10 == 0 or epoch == 1:
            print("    epoch {:2d}/{} | mean training loss {:.4f} | {:.0f}s".format(
                epoch, EPOCHS, mean_loss, time.time() - t0))
    torch.cuda.synchronize()
    wall = time.time() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 1024 ** 2
    probs = predict_fold9(model, arm)           # the ONE fold-9 evaluation, end of epoch 50
    files_sha = save_run_outputs(paths, probs, loss_rows, model)    # saved BEFORE any metric
    log_event(name, attempt, "completed", files_sha256=files_sha, wall_time_s=round(wall, 1),
              peak_vram_mb=round(peak_mb, 1), final_epoch_train_loss=loss_rows[-1]["mean_train_loss"])
    macro, per = macro_auroc_from_probs(probs)  # only after the files and the log entry exist
    del model, opt, sched, loader
    torch.cuda.empty_cache()
    return {"arch": arch, "arm": arm, "seed": seed, "lr": lr, "fold9_macro_auroc": macro,
            **{"auroc_" + sc: per[sc] for sc in SUPERCLASSES},
            "final_epoch_train_loss": loss_rows[-1]["mean_train_loss"],
            "wall_time_s": round(wall, 1), "peak_vram_mb": round(peak_mb, 1)}


# =====================================================================================
# PART B - REUSED v0.5 RUNS (no retraining)
# =====================================================================================
hdr("PART B - REUSED v0.5 RUNS (no retraining)")
LR_TABLE = pd.DataFrame(LR_SEL["table"])
GATE_RESULT = V05_MAN["sanity_gate"]["result"]
assert GATE_RESULT["arch"] == "resnet1d_wang" and GATE_RESULT["arm"] == "signal_only" and int(GATE_RESULT["seed"]) == 42 \
    and abs(float(GATE_RESULT["lr"]) - SELECTED_LR["resnet1d_wang"]) < 1e-15
CELL_ROWS = {}           # (arch, arm, seed) -> E1 row
CELL_PROBS = {}          # (arch, arm, seed) -> (n_fold9, 5)
for arch, arm, seed in REUSED_CELLS:
    lr = SELECTED_LR[arch]
    vp = v05_run_paths(arch, arm, seed, lr)
    probs = load_saved_probs(vp)
    macro, per = macro_auroc_from_probs(probs)
    if arm == "signal_plus_physio":
        ref_rows = LR_TABLE[(LR_TABLE["arch"] == arch) & (LR_TABLE["arm"] == arm) & (LR_TABLE["seed"] == seed)
                            & (np.abs(LR_TABLE["lr"] - lr) < 1e-15)]
        assert len(ref_rows) == 1
        ref = ref_rows.iloc[0].to_dict()
        ref_label = "lr_selection.json table"
    else:
        ref = GATE_RESULT
        ref_label = "v0.5_manifest.json sanity_gate.result"
    assert abs(macro - float(ref["fold9_macro_auroc"])) <= REUSE_TOL, \
        "{} {} seed {}: recomputed macro-AUROC {} != {} {}".format(arch, arm, seed, macro, ref_label, ref["fold9_macro_auroc"])
    for sc in SUPERCLASSES:
        assert abs(per[sc] - float(ref["auroc_" + sc])) <= REUSE_TOL, "{} {}: per-class AUROC {} differs".format(arch, arm, sc)
    loss_df = pd.read_csv(vp["loss"], encoding="utf-8-sig")
    assert len(loss_df) == EPOCHS
    final_loss = float(loss_df["mean_train_loss"].iloc[-1])
    assert abs(final_loss - float(ref["final_epoch_train_loss"])) <= 1e-9, "reused final-epoch loss differs from " + ref_label
    CELL_PROBS[(arch, arm, seed)] = probs
    CELL_ROWS[(arch, arm, seed)] = {
        "arch": arch, "arm": arm, "seed": seed, "lr": lr, "reused": True, "fold9_macro_auroc": macro,
        **{"auroc_" + sc: per[sc] for sc in SUPERCLASSES}, "final_epoch_train_loss": final_loss,
        "wall_time_s": float(ref.get("wall_time_s", float("nan"))), "peak_vram_mb": float(ref.get("peak_vram_mb", float("nan"))),
        "attempts": 0, "probs_file": vp["probs"], "probs_sha256": REUSED_FILES[os.path.basename(vp["probs"])]["sha256"]}
    print("  {:14s} {:19s} seed {:4d}: macro-AUROC {:.4f} == {} (|diff| {:.1e}) -> PASS".format(
        arch, arm, seed, macro, ref_label, abs(macro - float(ref["fold9_macro_auroc"]))))
print("reused cells enter the analysis as the seed-42 cells; they are referenced by path + sha256, not copied.")


# =====================================================================================
# PART C - EXPLORATORY ANALYSIS PLAN (write-once, BEFORE the first training run)
# =====================================================================================
hdr("PART C - EXPLORATORY ANALYSIS PLAN (write-once)")

RUN_ORDER = []
for arch in ARCHS:
    for seed in SEEDS:
        for arm in ARMS:
            if (arch, arm, seed) in REUSED_CELLS:
                continue
            RUN_ORDER.append((arch, arm, seed))
assert len(RUN_ORDER) == 15 and {a: sum(r[0] == a for r in RUN_ORDER) for a in ARCHS} == EXPECTED_NEW_RUNS

PART_E_TEXT = {
    "E1": ("per-run table: arch, arm, seed, reused (bool), macro-AUROC, 5 per-class AUROCs, final-epoch train loss "
           "(reused runs: from their v0.5 loss CSV), wall time, peak VRAM (reused: from lr_selection.json table / "
           "v0.5_manifest.json sanity_gate, else NaN). Saved as v0.7_runs_table.csv."),
    "E2": "seed summary per (arch, arm): mean, SD (ddof=1), min, max of macro-AUROC over the 3 seeds. Saved as v0.7_seed_summary.csv.",
    "E3": ("seed ensemble per (arch, arm): P_ens = arithmetic mean of the 3 seeds' probabilities per record and class; "
           "macro- and per-class AUROC. Saved as v0.7_ensemble_table.csv and v0.7_ensemble_{arch}_{arm}_fold9_probs.csv "
           "(ecg_id, patient_id, 5 probabilities)."),
    "E4": ("paired patient-level bootstrap on fold 9 with the reference implementation only: _pat9, _rows9 = "
           "REF['patient_rows_in_file_order'](fold-9 patient_id as int64); BOOT9 = REF['build_boot_idx'](len(_pat9), 2000, 7); "
           "assert len(_pat9) == 1942. Closed list (delta = B - A), for EACH architecture: C1 = signal_plus_all - "
           "signal_plus_physio; C2 = signal_plus_physio - signal_only. Each comparison (a) on the seed-ensemble "
           "probabilities [primary for v0.7] and (b) per seed (same seed in both arms) [secondary]; REF['paired_bootstrap'] "
           "and REF['summarize_deltas'] (point on the full fold 9, 95% and 90% percentile CIs, NaN-draw count); for (a) also "
           "per-class point deltas and 95% CIs from class_delta; SD (ddof=1) of the 3 per-seed point deltas. Saved as "
           "v0.7_bootstrap_summary.csv (arch x comparison x {ensemble, seed42, seed123, seed2024}) and "
           "v0.7_bootstrap_deltas_ensemble.csv (2000 x 4: arch x comparison). REF['h3_verdict'] is NOT called; no verdict "
           "string, no a/b/material flag and no 'within +/-0.01' label is computed or printed for fold 9."),
    "E5": BIAS_NOTE,
    "E6": ("untrained-column note: the fold-9 records that map to an all-zero training column (site=__UNKNOWN__) under "
           "the folds 1-8 encoder are reported as a COUNT only and asserted == 2 (final_run_protocol locked_quantities); "
           "no sensitivity analysis."),
    "E7": ("plausibility (WARN only, never stop, never retrain): any signal_only or signal_plus_physio run with fold-9 "
           "macro-AUROC < 0.90; any run whose final-epoch train loss is > 2x the median of the same arch."),
    "closed_list": "Anything not in E1-E7 is not computed in v0.7.",
}

PLAN = {
    "document": "v0.7 exploratory analysis plan (fold-9 development matrix; write-once, written before the first v0.7 training run)",
    "timestamp": datetime.now().astimezone().isoformat(),
    "code_version": CODE_VERSION,
    "status": "EXPLORATORY: no setting may be chosen, changed or re-run based on any v0.7 number",
    "parent_hashes": {k: {"self_sha256": DOC_SELF[k], "file_sha256": DOC_FILE[k]} for k in DOC_PATHS},
    "reference_implementation_sha256": REFERENCE_CODE_SHA256,
    "locked_stage_plan_v0.7": STAGE_PLAN_V07,
    "runs": [{"arch": a, "arm": m, "seed": s, "lr": SELECTED_LR[a], "reused_from_v0.5": (a, m, s) in REUSED_CELLS,
              "probs_file": (os.path.basename(v05_run_paths(a, m, s, SELECTED_LR[a])["probs"]) if (a, m, s) in REUSED_CELLS
                             else os.path.basename(run_paths(a, m, s, SELECTED_LR[a])["probs"]))}
             for a in ARCHS for m in ARMS for s in SEEDS],
    "run_order_new_runs": [run_name(a, m, s, SELECTED_LR[a]) for a, m, s in RUN_ORDER],
    "training": ("verbatim v0.5 train_run with the documented changes: FORBIDDEN_ARM guard removed; lr = SELECTED_LR[arch]; "
                 "after epoch 50: predict_fold9 -> save state_dict, train-loss CSV, probability CSV (atomic rename) -> log "
                 "'completed' with sha256 of the three files -> compute AUROC; head first-Linear widths 256/259/338 and "
                 "OneCycle total_steps 6850 asserted; recipe = dev_protocol.json + v0.5_training_details.json"),
    "crash_and_rerun_rule_locked": FRP["crash_and_rerun_rules"]["v0.7"],
    "crash_and_rerun_rule_implementation": (
        "append-only OUT_DIR/v0.7_attempt_log.jsonl: 'started' before a run, 'completed' (sha256 of probs/loss/state) after "
        "its files are saved, 'failed' (error message) on any exception or non-finite loss, then the exception is "
        "re-raised. On re-execution: probability CSV exists -> run COMPLETE, never re-run, hashes verified against the "
        "'completed' entry (if none: accept, log 'completed_recovered', WARN); no probability CSV and the log shows "
        "started/failed -> delete partial files of that run (logged as 'deleted_partial_files') and re-run from scratch "
        "with the same seed as attempt n+1; at most 2 attempts per run, then STOP and report without changing any setting."),
    "analyses": PART_E_TEXT,
    "bootstrap": {"N_BOOT": N_BOOT, "BOOT_SEED": BOOT_SEED, "fold9_patients": EXPECTED_FOLD9_PATIENTS,
                  "comparisons": {k: {"A": a, "B": b} for k, (a, b) in COMPARISONS.items()}},
    "self_sha256_procedure": SELF_SHA256_PROCEDURE,
    "write_once": True,
}
PLAN = plain(PLAN)
if PLAN_EXISTS:
    mine = {k: v for k, v in PLAN.items() if k != "timestamp"}
    disk = {k: v for k, v in PLAN_ON_DISK.items() if k not in ("timestamp", "self_sha256")}
    differing = sorted(set(mine) ^ set(disk)) + sorted(k for k in set(mine) & set(disk) if mine[k] != disk[k])
    assert not differing, "v0.7_exploratory_analysis_plan.json differs from this code in field(s) {} - aborting.".format(differing)
    PLAN_SHA = PLAN_ON_DISK["self_sha256"]
    print("exploratory analysis plan: already written (self_sha256 {}); identical except 'timestamp'.".format(PLAN_SHA))
else:
    PLAN_SHA = write_once_json(PLAN_PATH, PLAN)
    print("[saved] {} (self_sha256 {}, verified) -- BEFORE the first training run".format(PLAN_PATH, PLAN_SHA))
OUTPUT_FILES.append(PLAN_PATH)


# =====================================================================================
# PART D - TRAINING (15 new runs) with the crash / re-run rule
# =====================================================================================
hdr("PART D - TRAINING ({} new runs, lr {})".format(len(RUN_ORDER), SELECTED_LR))
ATTEMPT_LOG = read_attempt_log()
TRAINED_THIS_EXECUTION = []
for i_run, (arch, arm, seed) in enumerate(RUN_ORDER, start=1):
    lr = SELECTED_LR[arch]
    name = run_name(arch, arm, seed, lr)
    paths = run_paths(arch, arm, seed, lr)
    entries = [e for e in ATTEMPT_LOG if e.get("run") == name]
    n_started = sum(e.get("event") == "started" for e in entries)
    if os.path.isfile(paths["probs"]):
        # COMPLETE: the fold-9 evaluation exists -> never re-run
        done = [e for e in entries if e.get("event") in ("completed", "completed_recovered")]
        if done:
            rec = done[-1]["files_sha256"]
            for kind, p in paths.items():
                if rec.get(kind) is None:
                    continue
                assert os.path.isfile(p) and sha256_of(p) == rec[kind], \
                    "{}: {} file differs from its logged sha256 - aborting (files must not change).".format(name, kind)
            info = done[-1]
            print("  [{:2d}/15] {} -> COMPLETE (from attempt log; hashes verified; not re-run)".format(i_run, name))
        else:
            rec = {kind: (sha256_of(p) if os.path.isfile(p) else None) for kind, p in paths.items()}
            info = log_event(name, n_started, "completed_recovered", files_sha256=rec)
            warn("{}: probability CSV exists but the log has no 'completed' entry -> accepted as complete "
                 "('completed_recovered'); missing files: {}".format(name, [k for k, v in rec.items() if v is None]))
        probs = load_saved_probs(paths)
        macro, per = macro_auroc_from_probs(probs)
        final_loss = (float(pd.read_csv(paths["loss"], encoding="utf-8-sig")["mean_train_loss"].iloc[-1])
                      if os.path.isfile(paths["loss"]) else float("nan"))
        row = {"arch": arch, "arm": arm, "seed": seed, "lr": lr, "fold9_macro_auroc": macro,
               **{"auroc_" + sc: per[sc] for sc in SUPERCLASSES}, "final_epoch_train_loss": final_loss,
               "wall_time_s": float(info.get("wall_time_s", float("nan"))),
               "peak_vram_mb": float(info.get("peak_vram_mb", float("nan")))}
    else:
        if n_started >= MAX_ATTEMPTS:
            raise RuntimeError("{}: {} attempts already started and no fold-9 evaluation exists -> STOP and report "
                               "(crash rule: at most {} attempts; no setting may be changed).".format(name, n_started, MAX_ATTEMPTS))
        stale = [p for p in list(paths.values()) + [_partial(q) for q in paths.values()] if os.path.exists(p)]
        if stale:
            for p in stale:
                os.remove(p)
            log_event(name, n_started, "deleted_partial_files", files=[os.path.basename(p) for p in stale])
            note("{}: deleted partial files of the previous attempt: {}".format(name, [os.path.basename(p) for p in stale]))
        attempt = n_started + 1
        print("  [{:2d}/15] training {} | {} | seed {} | lr {:g} | attempt {} | {} epochs | batch {}".format(
            i_run, arch, arm, seed, lr, attempt, EPOCHS, BATCH_SIZE[arch]))
        log_event(name, attempt, "started")
        try:
            row = train_run(arch, arm, seed, lr, attempt)
        except BaseException as exc:
            log_event(name, attempt, "failed", error="{}: {}".format(type(exc).__name__, exc),
                      probability_file_exists=os.path.isfile(paths["probs"]))
            print("  {} attempt {} FAILED -> logged; {}".format(
                name, attempt, "re-run the cell to start attempt {}".format(attempt + 1) if attempt < MAX_ATTEMPTS
                else "STOP and report (no further attempts allowed)"))
            raise
        TRAINED_THIS_EXECUTION.append(name)
        print("  -> fold-9 macro-AUROC {:.4f} | {:.0f}s | peak VRAM {:.0f} MB".format(
            row["fold9_macro_auroc"], row["wall_time_s"], row["peak_vram_mb"]))
    n_att = sum(e.get("event") == "started" for e in ATTEMPT_LOG if e.get("run") == name)
    CELL_PROBS[(arch, arm, seed)] = load_saved_probs(paths)
    CELL_ROWS[(arch, arm, seed)] = {**row, "reused": False, "attempts": n_att, "probs_file": paths["probs"],
                                    "probs_sha256": sha256_of(paths["probs"])}
    for p in paths.values():
        if os.path.isfile(p):
            OUTPUT_FILES.append(p)
OUTPUT_FILES.append(ATTEMPT_LOG_PATH)
assert len(CELL_ROWS) == len(ARCHS) * len(ARMS) * len(SEEDS) == 18
print("all 18 cells available ({} trained in this execution).".format(len(TRAINED_THIS_EXECUTION)))
del X
gc.collect()


# =====================================================================================
# PART E - EXPLORATORY ANALYSIS (fold 9; all 18 cells; no decision is taken from it)
# =====================================================================================
hdr("PART E - EXPLORATORY ANALYSIS (fold 9) -- EXPLORATORY")
Y9 = lab.loc[mask_eval, SUPERCLASSES].to_numpy(dtype=np.int64)
FOLD9_ECG = current_index["ecg_id"].to_numpy()[eval_rows]
FOLD9_PID = current_index["patient_id"].to_numpy()[eval_rows]

# --- E1 per-run table ---
runs_df = pd.DataFrame([CELL_ROWS[(a, m, s)] for a in ARCHS for m in ARMS for s in SEEDS])
runs_df = runs_df[["arch", "arm", "seed", "lr", "reused", "fold9_macro_auroc"] + ["auroc_" + sc for sc in SUPERCLASSES]
                  + ["final_epoch_train_loss", "wall_time_s", "peak_vram_mb", "attempts", "probs_file", "probs_sha256"]]
save_csv(runs_df, PREFIX + "runs_table.csv", index=False)
print(runs_df.drop(columns=["lr", "probs_file", "probs_sha256"]).round(4).to_string(index=False))

# --- E2 seed summary ---
seed_rows = []
for a in ARCHS:
    for m in ARMS:
        v = runs_df[(runs_df["arch"] == a) & (runs_df["arm"] == m)]["fold9_macro_auroc"].to_numpy()
        assert len(v) == 3
        seed_rows.append({"arch": a, "arm": m, "n_seeds": 3, "mean": float(v.mean()), "sd_ddof1": float(v.std(ddof=1)),
                          "min": float(v.min()), "max": float(v.max())})
seed_df = pd.DataFrame(seed_rows)
save_csv(seed_df, PREFIX + "seed_summary.csv", index=False)

# --- E3 seed ensemble ---
ENS = {}
ens_rows = []
for a in ARCHS:
    for m in ARMS:
        P_ens = np.mean([CELL_PROBS[(a, m, s)] for s in SEEDS], axis=0)
        ENS[(a, m)] = P_ens
        macro, per = macro_auroc_from_probs(P_ens)
        ens_rows.append({"arch": a, "arm": m, "fold9_macro_auroc_ensemble": macro, **{"auroc_" + sc: per[sc] for sc in SUPERCLASSES}})
        edf = pd.DataFrame({"ecg_id": FOLD9_ECG, "patient_id": FOLD9_PID})
        for k, sc in enumerate(SUPERCLASSES):
            edf["prob_" + sc] = P_ens[:, k]
        save_csv(edf, "{}ensemble_{}_{}_fold9_probs.csv".format(PREFIX, a, m), index=False)
ens_df = pd.DataFrame(ens_rows)
save_csv(ens_df, PREFIX + "ensemble_table.csv", index=False)

# --- E6 untrained one-hot columns (count only) ---
_allzero18 = FRP["locked_quantities"]["categorical_encoder"]["allzero_training_columns_folds1to8"]
_dummies18 = {"device": device_dummies, "site": site_dummies, "nurse": nurse_dummies}
for col, dmy in _dummies18.items():
    for c in dmy.columns:
        assert (c in _allzero18) == (int(dmy.loc[mask_train, c].sum()) == 0), "all-zero training column set differs: " + c
E6_COUNTS = {col: int(dmy.loc[mask_eval, [c for c in dmy.columns if c in _allzero18]].to_numpy().sum())
             for col, dmy in _dummies18.items()}
assert E6_COUNTS == FRP["locked_quantities"]["fold9_records_on_untrained_columns_under_folds1to8_encoder"], E6_COUNTS
assert E6_COUNTS["site"] == 2 and int(site_dummies.loc[mask_eval, "site=__UNKNOWN__"].sum()) == 2

# --- E4 paired patient-level bootstrap (REF only) ---
t0 = time.time()
_pat9, _rows9 = REF["patient_rows_in_file_order"](FOLD9_PID.astype(np.int64))
assert len(_pat9) == EXPECTED_FOLD9_PATIENTS, "fold-9 patients {} != {}".format(len(_pat9), EXPECTED_FOLD9_PATIENTS)
BOOT9 = REF["build_boot_idx"](len(_pat9), N_BOOT, BOOT_SEED)
boot_rows_out = []
ens_delta_cols = {}
for a in ARCHS:
    for comp, (arm_a, arm_b) in COMPARISONS.items():
        inputs = [("ensemble", ENS[(a, arm_a)], ENS[(a, arm_b)])] + \
                 [("seed{}".format(s), CELL_PROBS[(a, arm_a, s)], CELL_PROBS[(a, arm_b, s)]) for s in SEEDS]
        comp_rows = []
        for label, P_a, P_b in inputs:
            md, cd = REF["paired_bootstrap"](Y9, P_a, P_b, _rows9, BOOT9)
            mac_a = REF["macro_auroc_strict"](Y9, P_a)
            mac_b = REF["macro_auroc_strict"](Y9, P_b)
            summ = REF["summarize_deltas"](mac_b - mac_a, md, NAN_DRAW_FLAG_FRACTION)
            row = {"arch": a, "comparison": comp, "arm_A": arm_a, "arm_B": arm_b, "input": label,
                   "macro_auroc_A": mac_a, "macro_auroc_B": mac_b, **summ}
            if label == "ensemble":
                ens_delta_cols["{}_{}".format(a, comp)] = md
                per_a = REF["per_class_auroc"](Y9, P_a)
                per_b = REF["per_class_auroc"](Y9, P_b)
                for k, sc in enumerate(SUPERCLASSES):
                    cs = REF["summarize_deltas"](per_b[k] - per_a[k], cd[:, k], NAN_DRAW_FLAG_FRACTION)
                    row.update({"class_delta_" + sc: cs["point_delta"], "class_lo95_" + sc: cs["lo95"],
                                "class_hi95_" + sc: cs["hi95"], "class_n_nan_" + sc: cs["n_nan"]})
            if summ["nan_flag"]:
                warn("{} {} {}: {} of {} bootstrap draws are NaN (> {:.0%})".format(a, comp, label, summ["n_nan"], N_BOOT,
                                                                                      NAN_DRAW_FLAG_FRACTION))
            comp_rows.append(row)
        sd_seed = float(np.std([r["point_delta"] for r in comp_rows if r["input"] != "ensemble"], ddof=1))
        for r in comp_rows:
            r["sd_ddof1_of_3_per_seed_point_deltas"] = sd_seed
        boot_rows_out.extend(comp_rows)
boot_df = pd.DataFrame(boot_rows_out)
save_csv(boot_df, PREFIX + "bootstrap_summary.csv", index=False)
save_csv(pd.DataFrame(ens_delta_cols), PREFIX + "bootstrap_deltas_ensemble.csv", index=False)
print("E4: {} paired bootstraps x {} draws in {:.0f}s (fold-9 patients {}, BOOT_SEED {})".format(
    len(boot_rows_out), N_BOOT, time.time() - t0, len(_pat9), BOOT_SEED))

# --- E7 plausibility (WARN only) ---
for _, r in runs_df.iterrows():
    if r["arm"] in ("signal_only", "signal_plus_physio") and r["fold9_macro_auroc"] < PLAUSIBILITY_MIN_AUROC:
        warn("E7: {} {} seed {}: fold-9 macro-AUROC {:.4f} < {:.2f}".format(r["arch"], r["arm"], r["seed"],
                                                                        r["fold9_macro_auroc"], PLAUSIBILITY_MIN_AUROC))
for a in ARCHS:
    sub = runs_df[runs_df["arch"] == a]
    med = float(sub["final_epoch_train_loss"].median())
    for _, r in sub.iterrows():
        if r["final_epoch_train_loss"] > LOSS_OUTLIER_FACTOR * med:
            warn("E7: {} {} seed {}: final-epoch train loss {:.4f} > {:.0f}x the {} median {:.4f}".format(
                a, r["arm"], r["seed"], r["final_epoch_train_loss"], LOSS_OUTLIER_FACTOR, a, med))

# --- printed tables (E2, E3, E5, E4, E6) ---
print("\n--- E2 seed summary (fold-9 macro-AUROC over seeds 42/123/2024) ---")
for _, r in seed_df.iterrows():
    print("  {:14s} {:19s} {:.4f} +/- {:.4f}  [min {:.4f}, max {:.4f}]".format(r["arch"], r["arm"], r["mean"],
                                                                             r["sd_ddof1"], r["min"], r["max"]))
print("\n--- E3 seed-ensemble (mean of 3 seeds' probabilities) ---")
print(ens_df.round(4).to_string(index=False))
print("\n[E5 BIAS NOTE] " + BIAS_NOTE)
print("\n--- E4 paired patient-level bootstrap, fold 9 (delta = B - A; numbers and intervals only) ---")
print("  {:14s} {:4s} {:9s} {:>8s}  {:>19s}  {:>19s}  {:>5s}".format("arch", "comp", "input", "point", "95% CI", "90% CI", "NaN"))
for _, r in boot_df.iterrows():
    print("  {:14s} {:4s} {:9s} {:+8.4f}  [{:+.4f}, {:+.4f}]  [{:+.4f}, {:+.4f}]  {:5d}".format(
        r["arch"], r["comparison"], r["input"], r["point_delta"], r["lo95"], r["hi95"], r["lo90"], r["hi90"], int(r["n_nan"])))
print("  C1 = signal_plus_all - signal_plus_physio | C2 = signal_plus_physio - signal_only")
print("  SD (ddof=1) of the 3 per-seed point deltas: " + " | ".join(
    "{} {} {:.4f}".format(r["arch"], r["comparison"], r["sd_ddof1_of_3_per_seed_point_deltas"])
    for _, r in boot_df[boot_df["input"] == "ensemble"].iterrows()))
print("  per-class point deltas (ensemble; 95% CI):")
for _, r in boot_df[boot_df["input"] == "ensemble"].iterrows():
    print("    {:14s} {}: ".format(r["arch"], r["comparison"]) + " | ".join(
        "{} {:+.4f} [{:+.4f}, {:+.4f}]".format(sc, r["class_delta_" + sc], r["class_lo95_" + sc], r["class_hi95_" + sc])
        for sc in SUPERCLASSES))
print("\n[E6] fold-9 records on an all-zero (untrained) one-hot column under the folds 1-8 encoder: {} "
      "(count only, == final_run_protocol locked_quantities; no sensitivity analysis)".format(E6_COUNTS))


# =====================================================================================
# PREREG_DIR UNCHANGED + MANIFEST
# =====================================================================================
hdr("MANIFEST")
assert prereg_snapshot() == PREREG_SNAPSHOT_START, "PREREG_DIR CHANGED during v0.7 - aborting (v0.7 must never write there)."
PREREG_UNCHANGED = True
assert ("wfdb" in sys.modules) == _WFDB_AT_START, "wfdb was imported by v0.7 - aborting."
ATTEMPT_LOG = read_attempt_log()
attempt_summary = {}
for a, m, s in RUN_ORDER:
    nm = run_name(a, m, s, SELECTED_LR[a])
    ev = [e for e in ATTEMPT_LOG if e.get("run") == nm]
    attempt_summary[nm] = {"started": sum(e["event"] == "started" for e in ev),
                           "failed": sum(e["event"] == "failed" for e in ev),
                           "completed": sum(e["event"] in ("completed", "completed_recovered") for e in ev),
                           "recovered": sum(e["event"] == "completed_recovered" for e in ev)}
GPU_WALL_NEW = float(runs_df.loc[~runs_df["reused"], "wall_time_s"].sum(min_count=1))
seen, outputs = set(), []
for p in OUTPUT_FILES:
    if p in seen or not os.path.isfile(p):
        continue
    seen.add(p)
    outputs.append({"file": p, "bytes": os.path.getsize(p), "sha256": sha256_of(p)})
manifest = {
    "code_version": CODE_VERSION,
    "status": "EXPLORATORY",
    "run_started": RUN_STARTED,
    "run_finished": datetime.now().astimezone().isoformat(),
    "versions": VERSIONS, "gpu": GPU_NAME,
    "input_files_sha256": INPUT_HASHES,
    "v0.4_signal_cache_sha256": CACHE_HASHES,
    "parent_documents": {k: {"self_sha256": DOC_SELF[k], "file_sha256": DOC_FILE[k]} for k in DOC_PATHS},
    "ots_files": OTS_FILES, "ots_note": OTS_NOTE,
    "earlier_manifests_sha256": {"v0.4_manifest.json": V04_MAN_SHA, "v0.5_manifest.json": V05_MAN_SHA,
                                 "v0.6_manifest.json": V06_MAN_SHA},
    "reference_implementation_sha256": REFERENCE_CODE_SHA256,
    "exploratory_analysis_plan_self_sha256": PLAN_SHA,
    "recipe_from_dev_protocol": {"epochs": EPOCHS, "weight_decay": WEIGHT_DECAY, "batch_size": BATCH_SIZE, "crop": CROP,
                                 "eval_stride": EVAL_STRIDE, "window_starts": WINDOW_STARTS, "aggregation": AGGREGATION},
    "selected_lr": SELECTED_LR,
    "env_regression": regression_rows,
    "reused_v0.5_files": REUSED_FILES,
    "attempt_log_summary": attempt_summary,
    "attempt_log_entries": ATTEMPT_LOG,
    "trained_this_execution": TRAINED_THIS_EXECUTION,
    "gpu_wall_time_new_runs_s": GPU_WALL_NEW,
    "bias_note": BIAS_NOTE,
    "fold9_records_on_untrained_columns": E6_COUNTS,
    "warnings": WARNINGS,
    "flags": {"fold10_opened": False, "prereg_dir_unchanged": PREREG_UNCHANGED, "h3_verdict_called": False},
    "outputs": outputs,
}
MANIFEST_PATH = os.path.join(OUT_DIR, PREFIX + "manifest.json")
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, default=_json_default)
for o in outputs:
    print("  {}  {}".format(o["sha256"], o["file"]))
print("[saved] " + MANIFEST_PATH)
print("manifest sha256:", sha256_of(MANIFEST_PATH))


# =====================================================================================
# FINAL SUMMARY
# =====================================================================================
hdr("SUMMARY - " + CODE_VERSION + " (EXPLORATORY)")
for k in DOC_PATHS:
    print("  {:18s} self {} | file {} | .ots present: PASS".format(k, DOC_SELF[k], DOC_FILE[k]))
print("environment regression     : PASS (scipy {}, sklearn {}, torch {})".format(
    VERSIONS["scipy"], VERSIONS["sklearn"], VERSIONS["torch"]))
print("reused v0.5 cells          : 3, AUROC recomputed == lr_selection.json / v0.5 manifest (tol {}): PASS".format(REUSE_TOL))
print("exploratory analysis plan  :", PLAN_SHA)
print("\nE2 fold-9 macro-AUROC, mean +/- SD over 3 seeds | E3 seed-ensemble:")
for a in ARCHS:
    for m in ARMS:
        r = seed_df[(seed_df["arch"] == a) & (seed_df["arm"] == m)].iloc[0]
        e = ens_df[(ens_df["arch"] == a) & (ens_df["arm"] == m)].iloc[0]
        print("  {:14s} {:19s} {:.4f} +/- {:.4f} | ensemble {:.4f}".format(a, m, r["mean"], r["sd_ddof1"],
                                                                          e["fold9_macro_auroc_ensemble"]))
print("\nE4 (delta = B - A; point | 95% CI | 90% CI):")
for _, r in boot_df.iterrows():
    print("  {:14s} {} {:9s} {:+.4f} | [{:+.4f}, {:+.4f}] | [{:+.4f}, {:+.4f}]".format(
        r["arch"], r["comparison"], r["input"], r["point_delta"], r["lo95"], r["hi95"], r["lo90"], r["hi90"]))
print("\n[E5] " + BIAS_NOTE)
print("\nwarnings ({}):".format(len(WARNINGS)))
for w in WARNINGS:
    print("  - " + w)
print("total GPU wall time, 15 new runs: {:.0f} s ({:.2f} h)".format(GPU_WALL_NEW, GPU_WALL_NEW / 3600.0)
      if math.isfinite(GPU_WALL_NEW) else "total GPU wall time, 15 new runs: n/a (recovered runs without timing)")
assert prereg_snapshot() == PREREG_SNAPSHOT_START, "PREREG_DIR CHANGED during v0.7 - aborting."
print("\nEXPLORATORY: no setting may be changed based on these numbers. Fold 10 was not opened. PREREG_DIR unchanged.")



STEP 0 - INTEGRITY
PREREG_DIR snapshot: 10 files
  babf101c1d69b322daef6907175fdc4c17b5a53755a68f13a998eefa0148dc2d  dev_protocol.json
  70c0ddb918108467cea97bddea0893ffa879aeeece8c941ea4fbdb46343a7a98  dev_protocol.json.ots
  e1a5260e72d2e8b0d27dc709639468b7f9a48a7c8e2f8af2d663e105dbd329d4  final_run_protocol.json
  585ea85d2d99dcb8c5060303def8dd8c20a5e72fa7f7df74b7d61fe66905719b  final_run_protocol.json.ots
  358d783b37bba6fcf61237a43817c1fd8594fe192f8ecdfca2c25b871a21415b  lr_selection.json
  7819486a0d2afe5cec2351fd71012d5c30512b6cf23c70acf4ae447419fa2463  lr_selection.json.ots
  11dd446272654fc80d65b2fdd40259fa674bbb419a44fde84e589b6e297cde32  pre_registration.json
  bb0c254734c52588bdfa5d4c1c4fa574eaa860f628c3f5c6b5402502acfba428  pre_registration.json.ots
  6480c191aeaed9ff1e6b1da5caa0c1e46908d453d30106a32beb7cf83678be11  v0.5_training_details.json
  497b7246a751a2d555aaee23f7c1b3ccf93b215893e13b3da82b7efdd59526d5  v0.5_training_details.json.ots
pre_registration   self 5183f545